In [1]:
import random
from functools import partial
from omegaconf import DictConfig
import os
import torch
import rasterio
import numpy as np
from pathlib import Path
import h5py
from typing import Dict, Any
import matplotlib.pyplot as plt
from omegaconf import OmegaConf

from dataloader_CIRCA.datasets import CIRCA_ADAPTED2UTILISE_Dataset
from typing import Any, Dict, Literal
from sklearn.metrics import r2_score
import math
import torch
import torchgeometry as tgm
from prodict import Prodict
from torch import Tensor
from lib.data_utils import extract_sample
from lib.data_utils import seed_worker, pad_collate
from lib import config_utils
from lib import data_utils
from lib import visutils # gallery / apply_brightness_factor / sequence2gallery 
from lib.eval_tools import (
    Imputation,
    visualize_att_for_one_head_across_time,
    visualize_att_for_target_t_across_heads
)
from inference_full_tile import Dataset_from_files
import json

import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

#### Utils functions

In [2]:
import matplotlib.pyplot as plt
import numpy as np
import matplotlib.patches as patches
import ipywidgets as widgets
from IPython.display import display, clear_output


def _to_cpu(x):
    """Copie récursive de tenseurs en CPU (détachés du graph)."""
    if isinstance(x, torch.Tensor):
        return x.detach().cpu().clone()
    elif isinstance(x, dict):
        return {k: _to_cpu(v) for k, v in x.items()}
    elif isinstance(x, (list, tuple)):
        return type(x)(_to_cpu(v) for v in x)
    return x


def sample_to_batch(sample: dict) -> dict:
    batch = {}
    for k, v in sample.items():
        if isinstance(v, torch.Tensor):
            batch[k] = v.unsqueeze(dim=0)
        else:
            batch[k] = v
    return batch


def plot_seq(batch, pred, c_index, BRIGHTNESS_FACTOR=3, t_sampled=None):
    if t_sampled is None:
        images = torch.concatenate(
            [
                batch['y'][0][:, c_index, :, :].swapaxes(1, 3).swapaxes(1, 2),
                batch['x'][0][:, c_index, :, :].swapaxes(1, 3).swapaxes(1, 2),
                pred[0][:, c_index, :, :].swapaxes(1, 3).swapaxes(1, 2),
            ],
            dim=0,
        )
    else:
        images = torch.concatenate(
            [
                batch['y'][0][:, c_index, :, :].swapaxes(1, 3).swapaxes(1, 2)[t_sampled],
                batch['x'][0][:, c_index, :, :].swapaxes(1, 3).swapaxes(1, 2)[t_sampled],
                pred[0][:, c_index, :, :].swapaxes(1, 3).swapaxes(1, 2)[t_sampled],
            ],
            dim=0,
        )
    images = visutils.apply_brightness_factor(images, factor=BRIGHTNESS_FACTOR)
    ncols = int(images.shape[0] / 3)
    fig, axes = plt.subplots(nrows=3, ncols=ncols, figsize=(15, 7))
    column_labels = [f't{i + 1}' for i in range(ncols + 1)]
    for idx, ax in enumerate(axes.flat):
        ax.imshow(images[idx])
        ax.axis('off')
        rect = patches.Rectangle((0, 0), 1, 1, 
                                   transform=ax.transAxes,
                                   fill=False, 
                                   edgecolor='black', 
                                   linewidth=1)
        ax.add_patch(rect)
        if idx < ncols:
            ax.set_title(column_labels[idx], fontsize=12, fontweight='bold')
    plt.subplots_adjust(wspace=0.05, hspace=0.1)
    
    row_labels = ['Target', 'Inputs', 'Predictions']
    for i, label in enumerate(row_labels):
        axes[i, 0].annotate(label, 
                            xy=(-0.1, 0.5),
                            xycoords='axes fraction',
                            fontsize=10, 
                            fontweight='bold',
                            ha='right',
                            va='center',
                            rotation=90)
    
    plt.subplots_adjust(left=0.18)
    plt.show()


def plot_seq_RGB_NIR(batch, pred, n_visible=8):
    # plot R-G-B
    plot_seq_interactive(batch, pred, c_index=[2, 1, 0], BRIGHTNESS_FACTOR=3, n_visible=n_visible)
    # plot NIR-R-G
    plot_seq_interactive(batch, pred, c_index=[6, 2, 1], BRIGHTNESS_FACTOR=2, n_visible=n_visible)

# ── Versions interactives avec slider ──────────────────────────────────────────

def _draw_grid(images, ncols, t_offset, row_labels, ax_array, fig, BRIGHTNESS_FACTOR):
    """Dessine la grille Target / Inputs / Predictions pour un sous-ensemble de timesteps."""
    images = visutils.apply_brightness_factor(images, factor=BRIGHTNESS_FACTOR)
    column_labels = [f't{t_offset + i}' for i in range(ncols)]
    for idx, ax in enumerate(ax_array.flat):
        ax.imshow(images[idx])
        ax.axis('off')
        rect = patches.Rectangle((0, 0), 1, 1,
                                 transform=ax.transAxes,
                                 fill=False, edgecolor='black', linewidth=1)
        ax.add_patch(rect)
        if idx < ncols:
            ax.set_title(column_labels[idx], fontsize=12, fontweight='bold')
    for i, label in enumerate(row_labels):
        ax_array[i, 0].annotate(label, xy=(-0.1, 0.5), xycoords='axes fraction',
                                fontsize=10, fontweight='bold',
                                ha='right', va='center', rotation=90)


def plot_seq_interactive(batch, pred, c_index, BRIGHTNESS_FACTOR=3, n_visible=8):
    """Version interactive de plot_seq avec un slider temporel.
    Les tenseurs sont copiés en CPU pour survivre à la suppression des variables d'origine."""
    batch = _to_cpu(batch)
    pred = _to_cpu(pred)

    target = batch['y'][0][:, c_index, :, :].swapaxes(1, 3).swapaxes(1, 2)
    inputs = batch['x'][0][:, c_index, :, :].swapaxes(1, 3).swapaxes(1, 2)
    preds  = pred[0][:, c_index, :, :].swapaxes(1, 3).swapaxes(1, 2)

    T = target.shape[0]
    n_visible = min(n_visible, T)

    out = widgets.Output()
    slider = widgets.IntSlider(
        value=0, min=0, max=max(T - n_visible, 0), step=1,
        description='t_start :',
        continuous_update=False,
        style={'description_width': 'initial'},
        layout=widgets.Layout(width='80%'),
    )
    label = widgets.Label(value=f'Série totale : {T} dates  |  Fenêtre : {n_visible}')

    def _update(_change=None):
        t0 = slider.value
        idx = slice(t0, t0 + n_visible)
        images = torch.concatenate([target[idx], inputs[idx], preds[idx]], dim=0)
        with out:
            clear_output(wait=True)
            fig, axes = plt.subplots(nrows=3, ncols=n_visible,
                                     figsize=(2.2 * n_visible, 7))
            _draw_grid(images, n_visible, t0,
                       ['Target', 'Inputs', 'Predictions'],
                       axes, fig, BRIGHTNESS_FACTOR)
            plt.subplots_adjust(left=0.10, wspace=0.05, hspace=0.1)
            plt.show()

    slider.observe(_update, names='value')
    display(widgets.VBox([label, slider, out]))
    _update()


def plot_seq_RGB_NIR_interactive(
    targets, 
    inputs,
    preds,
    n_visible=8):
    """Version interactive de plot_seq_RGB_NIR avec un slider partagé RGB / NIR.
    Les tenseurs sont copiés en CPU pour survivre à la suppression des variables d'origine."""
    targets = _to_cpu(targets)
    inputs = _to_cpu(inputs)
    preds = _to_cpu(preds)

    target_rgb = targets[:, [2, 1, 0], :, :].swapaxes(1, 3).swapaxes(1, 2)
    inputs_rgb = inputs[:, [2, 1, 0], :, :].swapaxes(1, 3).swapaxes(1, 2)
    preds_rgb  = preds[:, [2, 1, 0], :, :].swapaxes(1, 3).swapaxes(1, 2)

    target_nir = targets[:, [6, 2, 1], :, :].swapaxes(1, 3).swapaxes(1, 2)
    inputs_nir = inputs[:, [6, 2, 1], :, :].swapaxes(1, 3).swapaxes(1, 2)
    preds_nir  = preds[:, [6, 2, 1], :, :].swapaxes(1, 3).swapaxes(1, 2)

    T = target_rgb.shape[0]
    n_visible = min(n_visible, T)

    out = widgets.Output()
    slider = widgets.IntSlider(
        value=0, min=0, max=max(T - n_visible, 0), step=1,
        description='t_start :',
        continuous_update=False,
        style={'description_width': 'initial'},
        layout=widgets.Layout(width='80%'),
    )
    label = widgets.Label(value=f'Série totale : {T} dates  |  Fenêtre : {n_visible}')

    def _update(_change=None):
        t0 = slider.value
        idx = slice(t0, t0 + n_visible)

        imgs_rgb = torch.concatenate([target_rgb[idx], inputs_rgb[idx], preds_rgb[idx]], dim=0)
        imgs_nir = torch.concatenate([target_nir[idx], inputs_nir[idx], preds_nir[idx]], dim=0)

        row_labels = ['Target', 'Inputs', 'Predictions']
        with out:
            clear_output(wait=True)
            fig, axes = plt.subplots(nrows=6, ncols=n_visible,
                                     figsize=(2.2 * n_visible, 13))
            _draw_grid(imgs_rgb, n_visible, t0, row_labels,
                       axes[:3], fig, BRIGHTNESS_FACTOR=3)
            _draw_grid(imgs_nir, n_visible, t0, row_labels,
                       axes[3:], fig, BRIGHTNESS_FACTOR=2)

            # Titres de section
            fig.text(0.02, 0.78, 'RGB', fontsize=14, fontweight='bold',
                     rotation=90, va='center')
            fig.text(0.02, 0.35, 'NIR-R-G', fontsize=14, fontweight='bold',
                     rotation=90, va='center')

            plt.subplots_adjust(left=0.08, wspace=0.05, hspace=0.15)
            plt.show()

    slider.observe(_update, names='value')
    display(widgets.VBox([label, slider, out]))
    _update()

def getitem_from_mgrsc(df, mgrsc, window: str = None):
    if window is None:
        return df[df["mgrs25"] == mgrsc].index.values
    else:
        return df[(df["mgrs25"] == mgrsc) & (df["window"] == window)].index.values[0]

## Inférences faites depuis le fichier hdf5 

In [4]:
config_file = Path("./configs/config_run_train.yaml")
default_config_path = Path("./configs/default.yaml")
cfg_custom = config_utils.read_config(config_file)
cfg_default = config_utils.read_config(default_config_path)
config = OmegaConf.merge(cfg_default, cfg_custom)

# Faire différents changments dans les fichiers de configs pour les vérifications
config.data.hdf5_file = "/mnt/DATA_10T/data_rpg/circa/hdf5/CIRCA_CR_merged.hdf5"
config.mask.mask_type = 'random_fully_masked'
config.data.max_seq_length = None
config.training_settings.batch_size = 1
config.mask.ratio_masked_frames = 0.8
config.mask.intersect_real_cloud_masks = False
config.mask.dilate_cloud_masks = False

BRIGHTNESS_FACTOR = 3
phase = "test"

path_inference_config_file = Path("/home/SPeillet/rpg_3STR/cloud_reconstruction/U-TILISE/configs/config_run_eval.yaml")
# path_ckpt_config_file = Path("/mnt/DATA_10T/data_rpg/outputs/U-TILISE/results/ALL_SAR_120_epochs_2025-07-11_16-56/config.yaml")
# path_ckpt_pth = Path("/mnt/DATA_10T/data_rpg/outputs/U-TILISE/results/ALL_SAR_120_epochs_2025-07-11_16-56/checkpoints/Model_best.pth")

# path_ckpt_config_file = Path("/home/SPeillet/data_rpg/outputs/U-TILISE/results/multistream_all_bands_random_clouds/seq_length_12_bs_4_acc_iter_2_da_masked_0.6_fully_masked_0.2_epochs_65/config.yaml")
# path_ckpt_config_file = Path("/home/SPeillet/data_rpg/outputs/U-TILISE/results/multistream_all_bands_random_clouds/seq_length_12_bs_4_acc_iter_2_da_masked_0.6_fully_masked_0.2_epochs_65/checkpoints/Model_best.pth")

# path_ckpt_config_file = Path("/home/SPeillet/data_rpg/outputs/U-TILISE/results/all_bands_sar_closest_mix_da/config.yaml")
# path_ckpt_pth = Path("/home/SPeillet/data_rpg/outputs/U-TILISE/results/all_bands_sar_closest_mix_da/checkpoints/Model_best.pth")

path_ckpt_config_file = Path("/home/SPeillet/data_rpg/outputs/U-TILISE/results/all_bands_sar_closest_mix_random_fully_masked_da/2026-03-14_10-59/config.yaml")
path_ckpt_pth = Path("/home/SPeillet/data_rpg/outputs/U-TILISE/results/all_bands_sar_closest_mix_random_fully_masked_da/2026-03-14_10-59/checkpoints/Model_best.pth")

training_config_file = config_utils.read_config(path_ckpt_config_file)
inference_config_file = config_utils.read_config(path_inference_config_file)

temporal_window = 6
# Faire la différence entre l'imputation d'une TS (avec plusieurs intervalles de dates) et plusieurs observations.
inference_imputation = Imputation(
    config_file_train=path_inference_config_file, # Config permettant de faire la configuration de l'inference
    method="utilise",
    checkpoint=path_ckpt_pth,
    config_file_test=path_ckpt_config_file, # Fichier ayant servi à l'entrainement du modèle (update params model)
    temporal_window=temporal_window,
)

dset = data_utils.get_dataset(config, phase=phase)

# dset = torch.utils.data.Subset(dset, range(SUBSET_LENGTH))
dataloader = torch.utils.data.DataLoader(
    dataset=dset,
    batch_size=1,
    shuffle=False,
    num_workers=0,
    collate_fn=None,
    pin_memory=False,
    drop_last=False,
)



RuntimeError: Error(s) in loading state_dict for UTILISE:
	size mismatch for in_conv.conv.conv.0.weight: copying a param with shape torch.Size([64, 14, 3, 3]) from checkpoint, the shape in current model is torch.Size([64, 10, 3, 3]).
	size mismatch for out_conv.conv.conv.2.weight: copying a param with shape torch.Size([10, 64, 3, 3]) from checkpoint, the shape in current model is torch.Size([6, 64, 3, 3]).
	size mismatch for out_conv.conv.conv.2.bias: copying a param with shape torch.Size([10]) from checkpoint, the shape in current model is torch.Size([6]).

In [5]:
item = 500
t_sampled = [2, 4, 6, 8, 9, 10, 13]
t_masked = {"indices_masked": [1, 2, 4, 6]}
# t_sampled = [34, 35, 36, 37, 38, 39, 40, 41, 42]
# t_masked = None

# Faire différents changments dans les fichiers de configs pour les vérifications
config.data.hdf5_file = "/mnt/DATA_10T/data_rpg/circa/hdf5/CIRCA_CR_merged.hdf5"
config.mask.mask_type = 'random_fully_masked'
config.data.max_seq_length = None
config.training_settings.batch_size = 1
config.mask.ratio_masked_frames = 0.5
config.mask.intersect_real_cloud_masks = False
config.mask.dilate_cloud_masks = False

BRIGHTNESS_FACTOR = 3

dl1 = torch.utils.data.DataLoader(
    dataset=data_utils.get_dataset(config, phase="test"),
    batch_size=1,
    shuffle=False,
    num_workers=0,
    collate_fn=None,
    pin_memory=False,
    drop_last=False,
)

sample1 = dl1.dataset.__getitem__(item, t_sampled=t_sampled, t_masked=t_masked)
print(sample1["info"])
batch1 = sample_to_batch(sample1)

batch1, y_pred, att = inference_imputation.impute_sample(
    batch=batch1,
    return_att=True,
    # t_start=t_start,
    # t_end=t_end,
)

# plot_seq_RGB_NIR(batch1, y_pred)
plot_seq_RGB_NIR_interactive(
targets=batch1["y"][0],
inputs=batch1["x"][0],
preds=y_pred[0],
) 

{'mgrs': '30UXV', 'mgrs25': '30UXV_row-2_col-2', 'window': '1536_1940_256_256'}


NameError: name 'inference_imputation' is not defined

## Inférence faite depuis les fichiers du store-dai

### Good sample

In [5]:
store_dai = Path("/mnt/stores/store_dai")
path_dataset_circa = store_dai / "projets/pac/3str/EXP_2/Data_Raster"
data_optique = path_dataset_circa / "optique_dataset"
data_radar = path_dataset_circa / "radar_dataset_v4"
path_test_set_mgrs25 = store_dai / "projets/pac/3str/EXP_2/train_val_test/MGRSC_test.json"
test_mgrs25 = json.load(open(path_test_set_mgrs25))
# Faire différents changments dans les fichiers de configs pour les vérifications
config.data.hdf5_file = "/mnt/DATA_10T/data_rpg/circa/hdf5/CIRCA_CR_merged.hdf5"
config.mask.mask_type = 'random_fully_masked'
config.data.max_seq_length = None
config.training_settings.batch_size = 1
config.mask.ratio_masked_frames = 0.0
config.mask.intersect_real_cloud_masks = False
config.mask.dilate_cloud_masks = False

BRIGHTNESS_FACTOR = 3

dict_mgrs = {f.stem: f for f in data_optique.iterdir()}
dict_mgrsc = {p.stem: p for f in dict_mgrs.values() for p in f.iterdir()}

mgrsc_good_sample = '30UXV_row-2_col-2'
window_good_sample = (1536, 1940, 256, 256)
image_size = [256, 256]
OVERLAP = 0

ds_from_files = Dataset_from_files(
    mgrsc=mgrsc_good_sample,
    data_optique=data_optique,
    data_radar=data_radar,
    image_size=image_size,
    overlap=OVERLAP,
    fill_value=1.0,
    mask_type='original_masks',
    load_dataset="./tiles_windows.csv",
)

dl_from_files = torch.utils.data.DataLoader(
    dataset=ds_from_files,
    batch_size=1,
    shuffle=False,
    num_workers=0,
    collate_fn=None,
    pin_memory=False,
    drop_last=False,
)

In [6]:
index_good_sample = getitem_from_mgrsc(ds_from_files.mgrsc_dataset, mgrsc=mgrsc_good_sample, window=window_good_sample)
item = index_good_sample
# t_sampled = None
t_sampled = None
t_masked = None

sample_from_files = dl_from_files.dataset.__getitem__(item, t_sampled=[19, 20, 21, 22, 64, 65, 75]) #29
batch_from_files = sample_to_batch(sample_from_files)
# batch_from_files["position_days"] = batch_hdf5["position_days"]
# batch_from_files["masks_valid_obs"] = batch_hdf5["masks_valid_obs"]
batch_from_files, y_pred_from_files, att = inference_imputation.impute_sample(
    batch=batch_from_files,
    return_att=True,
    # t_start=t_start,
    # t_end=t_end,
)

plot_seq_RGB_NIR_interactive(
    targets=batch_from_files["y"][0],
    inputs=batch_from_files["x"][0],
    preds=y_pred_from_files[0],
    n_visible=8,
)

Original masks shape: torch.Size([7, 2, 256, 256])


In [7]:
sample_from_files = dl_from_files.dataset.__getitem__(item, t_sampled=t_sampled)
batch_from_files = sample_to_batch(sample_from_files)

batch_from_files, y_pred_from_files, att = inference_imputation.impute_sample(
    batch=batch_from_files,
    return_att=True,
    # t_start=t_start,
    # t_end=t_end,
)

plot_seq_RGB_NIR_interactive(
    targets=batch_from_files["y"][0], 
    inputs=batch_from_files["x"][0],
    preds=y_pred_from_files[0],
    n_visible=8,
)

Original masks shape: torch.Size([144, 2, 256, 256])


In [8]:
batch_from_files, y_pred_from_files, att = inference_imputation.impute_sample(
    batch=batch_from_files,
    return_att=True,
    # t_start=t_start,
    # t_end=t_end,
)

plot_seq_RGB_NIR_interactive(
    targets=batch_from_files["y"][0],
    inputs=batch_from_files["x"][0],
    preds=y_pred_from_files[0],
    n_visible=8,
)

In [ ]:
sample_from_files = dl_from_files.dataset.__getitem__(item, t_sampled=[19, 20, 64, 65, 75]) #29
batch_from_files = sample_to_batch(sample_from_files)

Original masks shape: torch.Size([5, 2, 256, 256])


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def analyze_and_plot_distributions(b_hdf5, b_tif):
    """
    Calcule et compare les distributions de pixels par grandes catégories
    (Optique S2 et potentiellement Radar S1 si présent) entre les deux approches.
    Rejette les pixels masqués pour ne juger que de la vraie donnée.
    """
    
    # 1. Extraction et nettoyage de 'y' (cibles complètes)
    # y est au format (Batch, Temps, Canaux, H, W). Pour la distribution globale on s'en fiche du temps et x,y.
    # On va regrouper tous les pixels valides.
    
    y_h5  = b_hdf5["y"][0].detach().cpu().numpy()
    y_tif = b_tif["y"][0].detach().cpu().numpy()

    c_s2 = 10 # Nombre de canaux Sentinel-2
    # Séparation Optique S2 (Les 10 premiers canaux)
    s2_h5  = y_h5[:, :c_s2, :, :].flatten()
    s2_tif = y_tif[:, :c_s2, :, :].flatten()
    
    print("="*60)
    print("STATISTIQUES GLOBALES - IMAGES CIBLES (y)")
    print("="*60)
    print(f"{'Source':<15} | {'Min':<8} | {'Max':<8} | {'Moyenne':<8} | {'Médiane':<8} | {'Ecart-type':<8}")
    print("-" * 60)
    print(f"{'HDF5 (S2)':<15} | {s2_h5.min():.4f}   | {s2_h5.max():.4f}   | {s2_h5.mean():.4f}   | {np.median(s2_h5):.4f}   | {s2_h5.std():.4f}")
    print(f"{'TIF_file (S2)':<15} | {s2_tif.min():.4f}   | {s2_tif.max():.4f}   | {s2_tif.mean():.4f}   | {np.median(s2_tif):.4f}   | {s2_tif.std():.4f}")
    
    # Si le radar est présent (Nb Canaux > 10)
    has_sar = y_h5.shape[1] > c_s2
    if has_sar:
        s1_h5 = y_h5[:, c_s2:, :, :].flatten()
        s1_tif = y_tif[:, c_s2:, :, :].flatten()
        print(f"{'HDF5 (S1_SAR)':<15} | {s1_h5.min():.4f}   | {s1_h5.max():.4f}   | {s1_h5.mean():.4f}   | {np.median(s1_h5):.4f}   | {s1_h5.std():.4f}")
        print(f"{'TIF_file(S1_SAR)':<15} | {s1_tif.min():.4f}   | {s1_tif.max():.4f}   | {s1_tif.mean():.4f}   | {np.median(s1_tif):.4f}   | {s1_tif.std():.4f}")
        
    print("\n")
        
    # 2. Visualisation des distributions de densité (Histogrammes)
    fig, axes = plt.subplots(1, 2 if has_sar else 1, figsize=(12 if has_sar else 6, 5))
    
    if not isinstance(axes, np.ndarray):
        axes = [axes]
        
    # Hist Optique
    # On limite à 1.0 au cas où, pour éviter que des valeurs extrêmes (outliers) écrasent le visuel
    bins_S2 = np.linspace(0, min(1.0, max(s2_h5.max(), s2_tif.max())), 100) 
    
    axes[0].hist(s2_h5, bins=bins_S2, alpha=0.5, density=True, label='HDF5 Pipeline', color='blue')
    axes[0].hist(s2_tif, bins=bins_S2, alpha=0.5, density=True, label='GeoTIFF Pipeline', color='orange')
    axes[0].set_title('Densité des valeurs des pixels Sentinel-2')
    axes[0].set_xlabel('Valeur normalisée (0 à 1)')
    axes[0].set_ylabel('Densité')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)

    # Hist Radar si applicable
    if has_sar:
        bins_S1 = np.linspace(0, 1.0, 100)
        axes[1].hist(s1_h5, bins=bins_S1, alpha=0.5, density=True, label='HDF5 Pipeline', color='blue')
        axes[1].hist(s1_tif, bins=bins_S1, alpha=0.5, density=True, label='GeoTIFF Pipeline', color='orange')
        axes[1].set_title('Densité des valeurs des pixels Sentinel-1 (SAR)')
        axes[1].set_xlabel('Valeur normalisée (0 à 1)')
        axes[1].legend()
        axes[1].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

# Exécution du test de compatibilité distributionnelle
analyze_and_plot_distributions(batch_hdf5, batch_from_files)

NameError: name 'batch_hdf5' is not defined

In [ ]:
batch_hdf5.keys()

In [ ]:
batch_hdf5["position_days"]

In [ ]:
batch_hdf5["position_days"]

### Observation direct des inférences en mode produit

In [ ]:
from rasterio.windows import Window
from dataloader_CIRCA.tools.mask_generation import masks_init_filling
import torch

def read_data_by_window(
    target_window,
    path_file,
):
    """
        Return a ndarray of a crop tile with shape T * H * W * C
    """
    with rasterio.open(path_file) as src:
        array = src.read(window=Window(*target_window))
        array = array.reshape((int(array.shape[0] // 12), 12, array.shape[1], array.shape[2])).astype(np.float32)
        return array[:, :10, ...], array[:, -1, ...]

In [ ]:
mgrsc_target = '30UXV_row-2_col-2'
target_window = (1536, 1940, 256, 256)
# mgrsc_target = mgrsc_bad_sample = "30TYS_row-4_col-4"
# target_window = window_bad_sample = (1536, 512, 256, 256)

path_preds = store_dai / "tmp/speillet/inferences"
pred_file = path_preds / f"pred_mgrsc_{mgrsc_target}.tif"
assert pred_file.exists(), f"File {pred_file} doesn't exists.."
preds, _ = read_data_by_window(
    target_window,
    path_file=pred_file,
)

path_inputs = store_dai / "projets/pac/3str/EXP_2/Data_Raster/optique_dataset"
input_file = path_inputs / mgrsc_target[:5] / f"MGRS25-{mgrsc_target}" /  f"bands_stacked_{mgrsc_target}.tif"
assert input_file.exists(), f"File {input_file} doesn't exists.."
inputs, cloud_mask = read_data_by_window(
    target_window,
    path_file=input_file,
)

inputs, preds, cloud_mask = torch.from_numpy(inputs), torch.from_numpy(preds), torch.from_numpy(cloud_mask)
inputs_masked, masks = masks_init_filling(
    seq=inputs.clone(),
    masks=cloud_mask.clone(),
    fill_type="fill_value",
    fill_value=1,
    dilate_cloud_masks=False,
)

In [ ]:
plot_seq_RGB_NIR_interactive(
    targets=inputs/10000, 
    inputs=inputs_masked/10000,
    preds=preds/10000,
    n_visible=8,
)

### Inférence depuis le hdf5

In [ ]:
import random
from functools import partial
from omegaconf import DictConfig
import os
import torch
import rasterio
import numpy as np
from pathlib import Path
import h5py
from typing import Dict, Any
import matplotlib.pyplot as plt
from omegaconf import OmegaConf
from lib import config_utils
from lib import data_utils
from lib import visutils # gallery / apply_brightness_factor / sequence2gallery 

from lib.eval_tools import (
    Imputation,
    visualize_att_for_one_head_across_time,
    visualize_att_for_target_t_across_heads
)

from dataloader_CIRCA.datasets import CIRCA_ADAPTED2UTILISE_Dataset
from lib.data_utils import pad_collate

from typing import Any, Dict, Literal
import math
import torch
from prodict import Prodict
from torch import Tensor
from lib.data_utils import extract_sample
from lib import config_utils
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)   # ou FutureWarning, UserWarning, ...

In [ ]:
def sample_to_batch(sample: dict) -> dict:
    batch = {}
    for k, v in sample.items():
        if isinstance(v, torch.Tensor):
            batch[k] = v.unsqueeze(dim=0)
        else:
            batch[k] = v
    return batch

In [ ]:
item = 500
# t_sampled = [2, 4, 6, 8, 9, 10, 13]
# t_masked = {"indices_masked": [1, 2, 4, 6]}

t_sampled = [0, 1, 2, 3, 4, 5, 6]
t_masked = {"indices_masked": [1, 2, 3, 4, 5]}

phase = "test"
BRIGHTNESS_FACTOR = 3
# Setup configuration
config_file = Path("./configs/config_run_train.yaml")
default_config_path = Path("./configs/default.yaml")
cfg_custom = config_utils.read_config(config_file)

cfg_default = config_utils.read_config(default_config_path)
config = OmegaConf.merge(cfg_default, cfg_custom)

config.data.hdf5_file = "/mnt/DATA_10T/data_rpg/circa/hdf5/CIRCA_CR_merged.hdf5"
config.mask.mask_type = 'random_fully_masked'
config.data.max_seq_length = None
config.training_settings.batch_size = 1
config.mask.ratio_masked_frames = 0.5
config.mask.intersect_real_cloud_masks = False
config.mask.dilate_cloud_masks = False

BRIGHTNESS_FACTOR = 3

dl_hdf5 = torch.utils.data.DataLoader(
    dataset=data_utils.get_dataset(config, phase="test"),
    batch_size=1,
    shuffle=False,
    num_workers=0,
    collate_fn=None,
    pin_memory=False,
    drop_last=False,
)

sample_hdf5 = dl_hdf5.dataset.__getitem__(item, t_sampled=t_sampled, t_masked=t_masked)
print(sample_hdf5["info"])

In [ ]:
batch_hdf5 = sample_to_batch(sample_hdf5)
batch_hdf5["position_days"] = batch_from_files["position_days"]
batch_hdf5, y_pred_hdf5, att = inference_imputation.impute_sample(
    batch=batch_hdf5,
    return_att=True,
    # t_start=t_start,
    # t_end=t_end,
)

plot_seq_RGB_NIR_interactive(
    targets=batch_hdf5["y"][0],
    inputs=batch_hdf5["x"][0],
    preds=y_pred_hdf5[0],
    n_visible=8,
)

In [ ]:
batch_from_files, y_pred_from_files, att = inference_imputation.impute_sample(
    batch=batch_from_files,
    return_att=True,
    # t_start=t_start,
    # t_end=t_end,
)

plot_seq_RGB_NIR_interactive(
    targets=batch_from_files["y"][0], 
    inputs=batch_from_files["x"][0],
    preds=y_pred_from_files[0],
    n_visible=8,
)

In [ ]:
batch_from_files["masks_valid_obs"]

In [ ]:
batch_hdf5["masks_valid_obs"]

In [ ]:
def stats(tensor):
    return f"Min={tensor.min().item():.3f}, Max={tensor.max().item():.3f}, Mean={tensor.float().mean().item():.3f}, Uniques={len(tensor.unique())}"

print("=== STATISTIQUES DES TENSEURS CRITIQUES ===")
for k in ["x", "y", "masks", "cloud_mask", "position_days", "days"]:
    print(f"\n--- {k.upper()} ---")
    print(f"HDF5 : {stats(batch_hdf5[k])}")
    print(f"TIF  : {stats(batch_from_files[k])}")
    if k in ["masks", "cloud_mask", "position_days", "days"]:
        print(f"HDF5 Jours/Vals : {batch_hdf5[k].flatten()[:10].tolist()}")
        print(f"TIF  Jours/Vals : {batch_from_files[k].flatten()[:10].tolist()}")

# Rapport de Métriques — Cloud Reconstruction U-TILISE

**Source des résultats :** logs SLURM dans `/mnt/stores/store_dai/tmp/speillet/logs`

## 1. Récapitulatif Global

### Masquage : Random Fully Masked

| Modèle | MAE | RMSE | PSNR | SSIM | SAM | R2 |
|:---|:---:|:---:|:---:|:---:|:---:|:---:|
| **ALL_SAR_120_epochs** (mix_closest) | 305.3 | 466.4 | 31.89 | 0.7129 | 0.0385 | 0.8487 |
| asc+desc, random_clouds, DA | 246.3 | 513.4 | 30.48 | 0.7515 | 0.0664 | 0.7896 |
| asc+desc, random_clouds | 239.6 | 512.5 | 30.83 | 0.7711 | 0.0695 | 0.7869 |
| asc+desc, random_fully_masked, DA | 332.9 | 534.7 | 31.30 | 0.7202 | 0.0404 | 0.8343 |
| mix_closest, random_fully_masked | 375.2 | 585.6 | 31.01 | 0.7167 | 0.0416 | 0.8306 |
| mix_closest, random_fully_masked, DA | 389.5 | 610.2 | 31.21 | 0.7357 | 0.0382 | 0.8353 |
| coherence_only | 243.9 | 504.7 | 30.23 | 0.7048 | 0.0696 | 0.7935 |
| **v2** mix_closest, random_fully_masked | 341.8 | 551.4 | 30.99 | 0.7234 | 0.0417 | 0.8305 |
| **v2** mix_closest, random_clouds | 294.5 | 462.0 | 32.20 | 0.7326 | 0.0353 | 0.8473 |
| **v2** asc+desc, random_fully_masked | 343.5 | 548.8 | 31.08 | 0.7200 | 0.0408 | 0.8313 |
| **v2** asc+desc, random_clouds | 303.6 | 469.4 | 31.73 | 0.7091 | 0.0392 | 0.8433 |

### Masquage : Consecutive Fully Masked

| Modèle | MAE | RMSE | PSNR | SSIM | SAM | R2 |
|:---|:---:|:---:|:---:|:---:|:---:|:---:|
| **ALL_SAR_120_epochs** (mix_closest) | 320.4 | 524.9 | 30.39 | 0.6548 | 0.0411 | 0.8197 |
| asc+desc, random_clouds, DA | 364.2 | 819.4 | 26.51 | 0.6881 | 0.0987 | 0.6286 |
| asc+desc, random_clouds | 355.0 | 810.5 | 26.95 | 0.7086 | 0.1284 | 0.6326 |
| asc+desc, random_fully_masked, DA | 339.4 | 576.2 | 30.56 | 0.6832 | 0.0395 | 0.8166 |
| mix_closest, random_fully_masked | 388.6 | 640.2 | 29.93 | 0.6772 | 0.0406 | 0.8054 |
| mix_closest, random_fully_masked, DA | 421.1 | 709.6 | 29.47 | 0.6861 | 0.0399 | 0.7864 |
| coherence_only | 371.0 | 826.0 | 26.16 | 0.6488 | 0.1002 | 0.6264 |
| **v2** mix_closest, random_fully_masked | 350.0 | 595.5 | 30.21 | 0.6783 | 0.0412 | 0.8110 |
| **v2** mix_closest, random_clouds | 302.2 | 508.3 | 31.09 | 0.6824 | 0.0360 | 0.8243 |
| **v2** asc+desc, random_fully_masked | 352.8 | 597.6 | 30.17 | 0.6760 | 0.0403 | 0.8087 |
| **v2** asc+desc, random_clouds | 311.7 | 513.8 | 30.69 | 0.6586 | 0.0393 | 0.8215 |

## 2. Métriques sur pixels reconstruits (Occluded)

### Masquage : Random Fully Masked

| Modèle | MAE | RMSE | PSNR | SSIM | SAM | R2 |
|:---|:---:|:---:|:---:|:---:|:---:|:---:|
| **ALL_SAR_120_epochs** (mix_closest) | 899.0 | 1108.3 | 22.54 | 0.4409 | 0.0886 | 0.6645 |
| asc+desc, random_clouds, DA | 998.2 | 1336.7 | 23.21 | 0.4328 | 0.2314 | 0.3926 |
| asc+desc, random_clouds | 983.2 | 1336.8 | 23.72 | 0.4275 | 0.2681 | 0.3750 |
| asc+desc, random_fully_masked, DA | 1055.5 | 1325.0 | 21.31 | 0.4019 | 0.1014 | 0.6231 |
| mix_closest, random_fully_masked | 1140.2 | 1412.7 | 21.09 | 0.4173 | 0.1036 | 0.6235 |
| mix_closest, random_fully_masked, DA | 1151.2 | 1440.3 | 21.14 | 0.4576 | 0.1010 | 0.6283 |
| coherence_only | 984.0 | 1314.7 | 22.04 | 0.3466 | 0.2360 | 0.4000 |
| **v2** mix_closest, random_fully_masked | 1085.4 | 1361.0 | 21.32 | 0.4587 | 0.0976 | 0.6275 |
| **v2** mix_closest, random_clouds | 900.9 | 1120.3 | 22.26 | 0.4566 | 0.0903 | 0.6500 |
| **v2** asc+desc, random_fully_masked | 1070.0 | 1343.3 | 21.33 | 0.4591 | 0.0980 | 0.6270 |
| **v2** asc+desc, random_clouds | 916.5 | 1128.8 | 22.04 | 0.4229 | 0.0965 | 0.6420 |

### Masquage : Consecutive Fully Masked

| Modèle | MAE | RMSE | PSNR | SSIM | SAM | R2 |
|:---|:---:|:---:|:---:|:---:|:---:|:---:|
| **ALL_SAR_120_epochs** (mix_closest) | 1081.9 | 1366.0 | 19.35 | 0.2190 | 0.1284 | 0.4716 |
| asc+desc, random_clouds, DA | 2296.9 | 2595.1 | 16.45 | 0.1560 | 0.6064 | 0.0721 |
| asc+desc, random_clouds | 2266.2 | 2570.1 | 17.09 | 0.1581 | 0.8594 | 0.0573 |
| asc+desc, random_fully_masked, DA | 1157.0 | 1512.9 | 19.39 | 0.2904 | 0.1122 | 0.5013 |
| mix_closest, random_fully_masked | 1305.4 | 1650.9 | 18.62 | 0.2937 | 0.1147 | 0.4997 |
| mix_closest, random_fully_masked, DA | 1437.6 | 1822.1 | 17.70 | 0.2879 | 0.1353 | 0.4253 |
| coherence_only | 2309.3 | 2608.6 | 15.12 | 0.1041 | 0.5966 | 0.1403 |
| **v2** mix_closest, random_fully_masked | 1195.6 | 1559.4 | 19.31 | 0.3089 | 0.1106 | 0.5010 |
| **v2** mix_closest, random_clouds | 1032.5 | 1335.7 | 19.74 | 0.2766 | 0.1151 | 0.4927 |
| **v2** asc+desc, random_fully_masked | 1192.1 | 1559.4 | 19.13 | 0.3149 | 0.1111 | 0.4921 |
| **v2** asc+desc, random_clouds | 1055.4 | 1341.4 | 19.59 | 0.2374 | 0.1191 | 0.4830 |

## 3. Occluded vs Observed (détail)

### Masquage : Random Fully Masked

| Modèle | Type | MAE | RMSE | PSNR | SSIM | SAM | R2 |
|:---|:---|:---:|:---:|:---:|:---:|:---:|:---:|
| **ALL_SAR_120_epochs** (mix_closest) | Occluded | 899.0 | 1108.3 | 22.54 | 0.4409 | 0.0886 | 0.6645 |
| **ALL_SAR_120_epochs** (mix_closest) | Observed | 69.2 | 99.0 | 40.19 | 0.8106 | 0.0288 | 0.9911 |
| asc+desc, random_clouds, DA | Occluded | 998.2 | 1336.7 | 23.21 | 0.4328 | 0.2314 | 0.3926 |
| asc+desc, random_clouds, DA | Observed | 64.3 | 93.8 | 40.71 | 0.8186 | 0.0271 | 0.9906 |
| asc+desc, random_clouds | Occluded | 983.2 | 1336.8 | 23.72 | 0.4275 | 0.2681 | 0.3750 |
| asc+desc, random_clouds | Observed | 56.1 | 82.1 | 41.83 | 0.8450 | 0.0234 | 0.9932 |
| asc+desc, random_fully_masked, DA | Occluded | 1055.5 | 1325.0 | 21.31 | 0.4019 | 0.1014 | 0.6231 |
| asc+desc, random_fully_masked, DA | Observed | 67.7 | 98.3 | 40.33 | 0.8276 | 0.0290 | 0.9891 |
| mix_closest, random_fully_masked | Occluded | 1140.2 | 1412.7 | 21.09 | 0.4173 | 0.1036 | 0.6235 |
| mix_closest, random_fully_masked | Observed | 71.1 | 102.9 | 39.94 | 0.8191 | 0.0300 | 0.9877 |
| mix_closest, random_fully_masked, DA | Occluded | 1151.2 | 1440.3 | 21.14 | 0.4576 | 0.1010 | 0.6283 |
| mix_closest, random_fully_masked, DA | Observed | 64.7 | 94.5 | 40.64 | 0.8339 | 0.0263 | 0.9912 |
| coherence_only | Occluded | 984.0 | 1314.7 | 22.04 | 0.3466 | 0.2360 | 0.4000 |
| coherence_only | Observed | 71.3 | 103.8 | 39.83 | 0.8044 | 0.0299 | 0.9883 |
| **v2** mix_closest, random_fully_masked | Occluded | 1085.4 | 1361.0 | 21.32 | 0.4587 | 0.0976 | 0.6275 |
| **v2** mix_closest, random_fully_masked | Observed | 73.3 | 106.3 | 39.63 | 0.8173 | 0.0311 | 0.9867 |
| **v2** mix_closest, random_clouds | Occluded | 900.9 | 1120.3 | 22.26 | 0.4566 | 0.0903 | 0.6500 |
| **v2** mix_closest, random_clouds | Observed | 59.7 | 85.5 | 41.51 | 0.8306 | 0.0249 | 0.9922 |
| **v2** asc+desc, random_fully_masked | Occluded | 1070.0 | 1343.3 | 21.33 | 0.4591 | 0.0980 | 0.6270 |
| **v2** asc+desc, random_fully_masked | Observed | 71.5 | 103.5 | 39.87 | 0.8123 | 0.0301 | 0.9875 |
| **v2** asc+desc, random_clouds | Occluded | 916.5 | 1128.8 | 22.04 | 0.4229 | 0.0965 | 0.6420 |
| **v2** asc+desc, random_clouds | Observed | 66.2 | 96.1 | 40.52 | 0.8095 | 0.0282 | 0.9894 |

### Masquage : Consecutive Fully Masked

| Modèle | Type | MAE | RMSE | PSNR | SSIM | SAM | R2 |
|:---|:---|:---:|:---:|:---:|:---:|:---:|:---:|
| **ALL_SAR_120_epochs** (mix_closest) | Occluded | 1081.9 | 1366.0 | 19.35 | 0.2190 | 0.1284 | 0.4716 |
| **ALL_SAR_120_epochs** (mix_closest) | Observed | 69.4 | 99.4 | 40.16 | 0.8098 | 0.0289 | 0.9910 |
| asc+desc, random_clouds, DA | Occluded | 2296.9 | 2595.1 | 16.45 | 0.1560 | 0.6064 | 0.0721 |
| asc+desc, random_clouds, DA | Observed | 64.5 | 94.2 | 40.67 | 0.8180 | 0.0271 | 0.9905 |
| asc+desc, random_clouds | Occluded | 2266.2 | 2570.1 | 17.09 | 0.1581 | 0.8594 | 0.0573 |
| asc+desc, random_clouds | Observed | 56.3 | 82.3 | 41.81 | 0.8445 | 0.0234 | 0.9932 |
| asc+desc, random_fully_masked, DA | Occluded | 1157.0 | 1512.9 | 19.39 | 0.2904 | 0.1122 | 0.5013 |
| asc+desc, random_fully_masked, DA | Observed | 67.9 | 98.5 | 40.31 | 0.8272 | 0.0290 | 0.9890 |
| mix_closest, random_fully_masked | Occluded | 1305.4 | 1650.9 | 18.62 | 0.2937 | 0.1147 | 0.4997 |
| mix_closest, random_fully_masked | Observed | 71.3 | 103.1 | 39.92 | 0.8185 | 0.0300 | 0.9877 |
| mix_closest, random_fully_masked, DA | Occluded | 1437.6 | 1822.1 | 17.70 | 0.2879 | 0.1353 | 0.4253 |
| mix_closest, random_fully_masked, DA | Observed | 65.1 | 94.9 | 40.60 | 0.8332 | 0.0264 | 0.9912 |
| coherence_only | Occluded | 2309.3 | 2608.6 | 15.12 | 0.1041 | 0.5966 | 0.1403 |
| coherence_only | Observed | 71.6 | 104.2 | 39.79 | 0.8038 | 0.0300 | 0.9883 |
| **v2** mix_closest, random_fully_masked | Occluded | 1195.6 | 1559.4 | 19.31 | 0.3089 | 0.1106 | 0.5010 |
| **v2** mix_closest, random_fully_masked | Observed | 73.7 | 106.7 | 39.60 | 0.8166 | 0.0311 | 0.9867 |
| **v2** mix_closest, random_clouds | Occluded | 1032.5 | 1335.7 | 19.74 | 0.2766 | 0.1151 | 0.4927 |
| **v2** mix_closest, random_clouds | Observed | 59.9 | 85.8 | 41.47 | 0.8300 | 0.0249 | 0.9922 |
| **v2** asc+desc, random_fully_masked | Occluded | 1192.1 | 1559.4 | 19.13 | 0.3149 | 0.1111 | 0.4921 |
| **v2** asc+desc, random_fully_masked | Observed | 71.9 | 104.1 | 39.82 | 0.8117 | 0.0302 | 0.9875 |
| **v2** asc+desc, random_clouds | Occluded | 1055.4 | 1341.4 | 19.59 | 0.2374 | 0.1191 | 0.4830 |
| **v2** asc+desc, random_clouds | Observed | 66.4 | 96.4 | 40.48 | 0.8088 | 0.0283 | 0.9893 |

## 4. Métriques par bande spectrale

### SSIM par bande

#### Masquage : Random Fully Masked

| Modèle | B2 | B3 | B4 | B5 | B6 | B7 | B8 | B8A | B11 | B12 |
|:---|:---:|:---:|:---:|:---:|:---:|:---:|:---:|:---:|:---:|:---:|
| **ALL_SAR_120_epochs** (mix_closest) | 0.5482 | 0.6355 | 0.6518 | 0.7183 | 0.7721 | 0.7923 | 0.7425 | 0.7917 | 0.7488 | 0.7274 |
| asc+desc, random_clouds, DA | 0.6030 | 0.6742 | 0.6824 | 0.7641 | 0.8065 | 0.8238 | 0.7683 | 0.8239 | 0.7958 | 0.7724 |
| asc+desc, random_clouds | 0.6465 | 0.7092 | 0.7203 | 0.7751 | 0.8118 | 0.8299 | 0.8036 | 0.8282 | 0.8044 | 0.7825 |
| asc+desc, random_fully_masked, DA | 0.5688 | 0.6508 | 0.6544 | 0.7270 | 0.7681 | 0.7863 | 0.7510 | 0.7888 | 0.7631 | 0.7442 |
| mix_closest, random_fully_masked | 0.5599 | 0.6461 | 0.6525 | 0.7181 | 0.7644 | 0.7838 | 0.7468 | 0.7834 | 0.7687 | 0.7437 |
| mix_closest, random_fully_masked, DA | 0.5957 | 0.6732 | 0.6804 | 0.7427 | 0.7731 | 0.7946 | 0.7687 | 0.7915 | 0.7780 | 0.7596 |
| coherence_only | 0.5442 | 0.6173 | 0.6230 | 0.7065 | 0.7719 | 0.7916 | 0.7376 | 0.7951 | 0.7420 | 0.7191 |
| **v2** mix_closest, random_fully_masked | 0.5774 | 0.6585 | 0.6678 | 0.7288 | 0.7734 | 0.7890 | 0.7387 | 0.7908 | 0.7675 | 0.7420 |
| **v2** mix_closest, random_clouds | 0.5842 | 0.6628 | 0.6713 | 0.7335 | 0.7743 | 0.7943 | 0.7690 | 0.7974 | 0.7811 | 0.7580 |
| **v2** asc+desc, random_fully_masked | 0.5678 | 0.6434 | 0.6491 | 0.7279 | 0.7717 | 0.7879 | 0.7441 | 0.7898 | 0.7709 | 0.7477 |
| **v2** asc+desc, random_clouds | 0.5587 | 0.6341 | 0.6423 | 0.7142 | 0.7663 | 0.7840 | 0.7273 | 0.7846 | 0.7495 | 0.7301 |

#### Masquage : Consecutive Fully Masked

| Modèle | B2 | B3 | B4 | B5 | B6 | B7 | B8 | B8A | B11 | B12 |
|:---|:---:|:---:|:---:|:---:|:---:|:---:|:---:|:---:|:---:|:---:|
| **ALL_SAR_120_epochs** (mix_closest) | 0.5055 | 0.5826 | 0.5994 | 0.6569 | 0.7083 | 0.7285 | 0.6835 | 0.7273 | 0.6881 | 0.6678 |
| asc+desc, random_clouds, DA | 0.5554 | 0.6186 | 0.6281 | 0.6980 | 0.7363 | 0.7528 | 0.7033 | 0.7521 | 0.7281 | 0.7087 |
| asc+desc, random_clouds | 0.5994 | 0.6545 | 0.6665 | 0.7108 | 0.7423 | 0.7595 | 0.7379 | 0.7569 | 0.7384 | 0.7200 |
| asc+desc, random_fully_masked, DA | 0.5407 | 0.6169 | 0.6228 | 0.6859 | 0.7279 | 0.7461 | 0.7141 | 0.7472 | 0.7246 | 0.7061 |
| mix_closest, random_fully_masked | 0.5298 | 0.6106 | 0.6189 | 0.6737 | 0.7221 | 0.7415 | 0.7086 | 0.7403 | 0.7242 | 0.7021 |
| mix_closest, random_fully_masked, DA | 0.5520 | 0.6249 | 0.6344 | 0.6876 | 0.7230 | 0.7455 | 0.7217 | 0.7414 | 0.7230 | 0.7078 |
| coherence_only | 0.5041 | 0.5698 | 0.5769 | 0.6494 | 0.7073 | 0.7260 | 0.6789 | 0.7282 | 0.6827 | 0.6643 |
| **v2** mix_closest, random_fully_masked | 0.5387 | 0.6136 | 0.6260 | 0.6801 | 0.7267 | 0.7429 | 0.6954 | 0.7442 | 0.7194 | 0.6962 |
| **v2** mix_closest, random_clouds | 0.5415 | 0.6141 | 0.6252 | 0.6795 | 0.7219 | 0.7421 | 0.7185 | 0.7446 | 0.7291 | 0.7078 |
| **v2** asc+desc, random_fully_masked | 0.5310 | 0.6009 | 0.6096 | 0.6804 | 0.7251 | 0.7424 | 0.7015 | 0.7430 | 0.7232 | 0.7029 |
| **v2** asc+desc, random_clouds | 0.5176 | 0.5865 | 0.5984 | 0.6581 | 0.7114 | 0.7295 | 0.6784 | 0.7302 | 0.6958 | 0.6799 |

### PSNR par bande

#### Masquage : Random Fully Masked

| Modèle | B2 | B3 | B4 | B5 | B6 | B7 | B8 | B8A | B11 | B12 |
|:---|:---:|:---:|:---:|:---:|:---:|:---:|:---:|:---:|:---:|:---:|
| **ALL_SAR_120_epochs** (mix_closest) | 35.06 | 35.77 | 33.97 | 33.76 | 31.65 | 31.43 | 29.28 | 30.87 | 32.36 | 32.32 |
| asc+desc, random_clouds, DA | 34.46 | 34.39 | 32.76 | 32.99 | 30.22 | 29.85 | 27.67 | 29.22 | 30.95 | 31.12 |
| asc+desc, random_clouds | 35.51 | 34.84 | 33.39 | 33.45 | 30.30 | 29.91 | 28.00 | 29.27 | 31.08 | 31.46 |
| asc+desc, random_fully_masked, DA | 34.53 | 35.53 | 33.33 | 33.43 | 30.97 | 30.64 | 28.76 | 30.10 | 32.23 | 32.08 |
| mix_closest, random_fully_masked | 34.17 | 35.18 | 32.95 | 33.01 | 30.67 | 30.29 | 28.58 | 29.82 | 31.83 | 31.84 |
| mix_closest, random_fully_masked, DA | 34.67 | 35.39 | 33.37 | 34.09 | 30.72 | 30.25 | 28.56 | 29.68 | 31.72 | 32.40 |
| coherence_only | 33.87 | 34.31 | 32.47 | 32.24 | 30.00 | 29.69 | 27.70 | 29.14 | 30.42 | 30.79 |
| **v2** mix_closest, random_fully_masked | 34.26 | 35.33 | 33.18 | 33.33 | 30.77 | 30.20 | 28.37 | 29.74 | 31.82 | 31.70 |
| **v2** mix_closest, random_clouds | 35.71 | 36.15 | 34.27 | 34.64 | 31.80 | 31.31 | 29.45 | 30.83 | 32.80 | 33.35 |
| **v2** asc+desc, random_fully_masked | 34.39 | 35.30 | 33.11 | 33.26 | 30.80 | 30.28 | 28.52 | 29.87 | 31.86 | 31.93 |
| **v2** asc+desc, random_clouds | 35.40 | 35.81 | 33.53 | 34.03 | 31.81 | 31.00 | 28.98 | 30.64 | 32.29 | 32.27 |

#### Masquage : Consecutive Fully Masked

| Modèle | B2 | B3 | B4 | B5 | B6 | B7 | B8 | B8A | B11 | B12 |
|:---|:---:|:---:|:---:|:---:|:---:|:---:|:---:|:---:|:---:|:---:|
| **ALL_SAR_120_epochs** (mix_closest) | 32.68 | 33.40 | 32.22 | 31.86 | 30.19 | 30.04 | 28.08 | 29.44 | 31.22 | 31.33 |
| asc+desc, random_clouds, DA | 29.70 | 29.59 | 28.78 | 28.41 | 26.04 | 25.89 | 23.90 | 25.19 | 26.85 | 27.59 |
| asc+desc, random_clouds | 30.62 | 30.08 | 29.43 | 28.97 | 26.28 | 26.11 | 24.35 | 25.41 | 27.10 | 27.96 |
| asc+desc, random_fully_masked, DA | 32.45 | 33.60 | 32.13 | 32.01 | 30.26 | 30.19 | 28.35 | 29.60 | 31.70 | 31.76 |
| mix_closest, random_fully_masked | 31.80 | 32.84 | 31.48 | 31.17 | 29.47 | 29.43 | 27.84 | 28.99 | 30.99 | 31.23 |
| mix_closest, random_fully_masked, DA | 31.45 | 32.21 | 30.95 | 31.39 | 29.19 | 29.03 | 27.44 | 28.44 | 29.57 | 30.87 |
| coherence_only | 29.13 | 29.34 | 28.41 | 27.64 | 25.67 | 25.59 | 23.80 | 24.99 | 26.29 | 27.22 |
| **v2** mix_closest, random_fully_masked | 32.14 | 33.29 | 31.83 | 31.77 | 30.05 | 29.73 | 27.95 | 29.23 | 31.19 | 31.29 |
| **v2** mix_closest, random_clouds | 33.48 | 34.07 | 32.86 | 33.02 | 30.68 | 30.39 | 28.63 | 29.85 | 32.02 | 32.79 |
| **v2** asc+desc, random_fully_masked | 32.12 | 33.15 | 31.71 | 31.58 | 29.97 | 29.75 | 28.03 | 29.30 | 30.96 | 31.22 |
| **v2** asc+desc, random_clouds | 33.21 | 33.72 | 32.16 | 32.38 | 30.75 | 30.17 | 28.24 | 29.75 | 31.48 | 31.72 |

### MAE par bande

#### Masquage : Random Fully Masked

| Modèle | B2 | B3 | B4 | B5 | B6 | B7 | B8 | B8A | B11 | B12 |
|:---|:---:|:---:|:---:|:---:|:---:|:---:|:---:|:---:|:---:|:---:|
| **ALL_SAR_120_epochs** (mix_closest) | 198.8 | 203.5 | 221.7 | 259.0 | 354.6 | 380.7 | 432.1 | 407.5 | 323.4 | 271.4 |
| asc+desc, random_clouds, DA | 159.2 | 165.7 | 184.5 | 200.6 | 283.0 | 304.8 | 352.0 | 326.4 | 261.0 | 225.7 |
| asc+desc, random_clouds | 142.9 | 160.0 | 174.2 | 193.5 | 280.2 | 301.5 | 341.7 | 323.5 | 259.7 | 219.1 |
| asc+desc, random_fully_masked, DA | 228.3 | 227.0 | 251.6 | 281.5 | 386.4 | 413.8 | 461.8 | 441.1 | 340.0 | 297.0 |
| mix_closest, random_fully_masked | 265.6 | 266.0 | 292.8 | 327.7 | 432.3 | 461.4 | 503.6 | 482.5 | 384.3 | 335.6 |
| mix_closest, random_fully_masked, DA | 283.4 | 287.3 | 315.3 | 336.5 | 425.3 | 459.0 | 516.9 | 495.4 | 427.8 | 347.8 |
| coherence_only | 165.6 | 161.0 | 184.1 | 204.9 | 278.1 | 296.9 | 342.3 | 317.5 | 262.9 | 225.7 |
| **v2** mix_closest, random_fully_masked | 240.0 | 234.5 | 262.0 | 287.4 | 387.7 | 422.6 | 470.4 | 445.9 | 354.1 | 313.2 |
| **v2** mix_closest, random_clouds | 184.8 | 191.9 | 210.6 | 238.2 | 348.9 | 379.5 | 422.8 | 401.6 | 311.5 | 255.7 |
| **v2** asc+desc, random_fully_masked | 232.7 | 231.5 | 260.8 | 292.1 | 391.9 | 424.1 | 470.0 | 448.8 | 367.8 | 315.6 |
| **v2** asc+desc, random_clouds | 194.7 | 201.3 | 231.4 | 252.8 | 343.6 | 381.0 | 427.3 | 402.0 | 323.5 | 278.5 |

#### Masquage : Consecutive Fully Masked

| Modèle | B2 | B3 | B4 | B5 | B6 | B7 | B8 | B8A | B11 | B12 |
|:---|:---:|:---:|:---:|:---:|:---:|:---:|:---:|:---:|:---:|:---:|
| **ALL_SAR_120_epochs** (mix_closest) | 213.9 | 217.7 | 235.3 | 273.9 | 370.6 | 397.8 | 450.4 | 426.7 | 334.8 | 282.7 |
| asc+desc, random_clouds, DA | 242.5 | 257.6 | 269.6 | 306.9 | 419.6 | 449.9 | 502.3 | 481.8 | 389.9 | 321.5 |
| asc+desc, random_clouds | 224.1 | 249.8 | 257.0 | 297.0 | 415.3 | 444.6 | 488.7 | 475.6 | 384.1 | 313.9 |
| asc+desc, random_fully_masked, DA | 243.4 | 239.8 | 260.8 | 293.1 | 392.3 | 414.7 | 463.1 | 443.0 | 344.6 | 299.5 |
| mix_closest, random_fully_masked | 285.6 | 285.4 | 307.4 | 347.0 | 448.4 | 471.3 | 512.8 | 491.5 | 393.1 | 343.5 |
| mix_closest, random_fully_masked, DA | 321.4 | 324.7 | 351.8 | 373.8 | 451.2 | 480.1 | 539.6 | 518.3 | 474.1 | 376.3 |
| coherence_only | 255.1 | 259.8 | 275.2 | 319.7 | 427.5 | 455.3 | 504.8 | 485.5 | 399.1 | 328.0 |
| **v2** mix_closest, random_fully_masked | 256.0 | 248.9 | 274.0 | 301.2 | 394.5 | 424.8 | 473.1 | 448.9 | 361.1 | 318.2 |
| **v2** mix_closest, random_clouds | 194.6 | 200.9 | 218.1 | 247.1 | 358.1 | 387.1 | 430.6 | 411.0 | 316.9 | 258.0 |
| **v2** asc+desc, random_fully_masked | 250.4 | 246.9 | 273.1 | 307.1 | 399.5 | 426.2 | 472.9 | 451.9 | 376.7 | 322.9 |
| **v2** asc+desc, random_clouds | 205.4 | 212.2 | 240.8 | 263.8 | 351.9 | 387.2 | 434.2 | 409.6 | 329.4 | 282.7 |

## 5. Disponibilité des résultats

| Modèle | Random Fully Masked | Consecutive Fully Masked |
|:---|:---:|:---:|
| **ALL_SAR_120_epochs** (mix_closest) | ✅ | ✅ |
| asc+desc, random_clouds, DA | ✅ | ✅ |
| asc+desc, random_clouds | ✅ | ✅ |
| asc+desc, random_fully_masked, DA | ✅ | ✅ |
| mix_closest, random_fully_masked | ✅ | ✅ |
| mix_closest, random_fully_masked, DA | ✅ | ✅ |
| coherence_only | ✅ | ✅ |
| **v2** mix_closest, random_fully_masked | ✅ | ✅ |
| **v2** mix_closest, random_clouds | ✅ | ✅ |
| **v2** asc+desc, random_fully_masked | ✅ | ✅ |
| **v2** asc+desc, random_clouds | ✅ | ✅ |


# Rapport de Métriques — Cloud Reconstruction U-TILISE

**Source des résultats :** logs SLURM dans `/mnt/stores/store_dai/tmp/speillet/logs`

## 1. Récapitulatif Global

### Masquage : Random Fully Masked

| Modèle | MAE | RMSE | PSNR | SSIM | SAM | R2 |
|:---|:---:|:---:|:---:|:---:|:---:|:---:|
| **ALL_SAR_120_epochs** (mix_closest) | 305.3 | 466.4 | 31.89 | 0.7129 | 0.0385 | 0.8487 |
| asc+desc, random_clouds, DA | 246.3 | 513.4 | 30.48 | 0.7515 | 0.0664 | 0.7896 |
| asc+desc, random_clouds | 239.6 | 512.5 | 30.83 | 0.7711 | 0.0695 | 0.7869 |
| asc+desc, random_fully_masked, DA | 332.9 | 534.7 | 31.30 | 0.7202 | 0.0404 | 0.8343 |
| mix_closest, random_fully_masked | 375.2 | 585.6 | 31.01 | 0.7167 | 0.0416 | 0.8306 |
| mix_closest, random_fully_masked, DA | 389.5 | 610.2 | 31.21 | 0.7357 | 0.0382 | 0.8353 |
| coherence_only | 243.9 | 504.7 | 30.23 | 0.7048 | 0.0696 | 0.7935 |
| **v2** mix_closest, random_fully_masked | 341.8 | 551.4 | 30.99 | 0.7234 | 0.0417 | 0.8305 |
| **v2** mix_closest, random_clouds | 294.5 | 462.0 | 32.20 | 0.7326 | 0.0353 | 0.8473 |
| **v2** asc+desc, random_fully_masked | 343.5 | 548.8 | 31.08 | 0.7200 | 0.0408 | 0.8313 |
| **v2** asc+desc, random_clouds | 303.6 | 469.4 | 31.73 | 0.7091 | 0.0392 | 0.8433 |

### Masquage : Consecutive Fully Masked

| Modèle | MAE | RMSE | PSNR | SSIM | SAM | R2 |
|:---|:---:|:---:|:---:|:---:|:---:|:---:|
| **ALL_SAR_120_epochs** (mix_closest) | 320.4 | 524.9 | 30.39 | 0.6548 | 0.0411 | 0.8197 |
| asc+desc, random_clouds, DA | 364.2 | 819.4 | 26.51 | 0.6881 | 0.0987 | 0.6286 |
| asc+desc, random_clouds | 355.0 | 810.5 | 26.95 | 0.7086 | 0.1284 | 0.6326 |
| asc+desc, random_fully_masked, DA | 339.4 | 576.2 | 30.56 | 0.6832 | 0.0395 | 0.8166 |
| mix_closest, random_fully_masked | 388.6 | 640.2 | 29.93 | 0.6772 | 0.0406 | 0.8054 |
| mix_closest, random_fully_masked, DA | 421.1 | 709.6 | 29.47 | 0.6861 | 0.0399 | 0.7864 |
| coherence_only | 371.0 | 826.0 | 26.16 | 0.6488 | 0.1002 | 0.6264 |
| **v2** mix_closest, random_fully_masked | 350.0 | 595.5 | 30.21 | 0.6783 | 0.0412 | 0.8110 |
| **v2** mix_closest, random_clouds | 302.2 | 508.3 | 31.09 | 0.6824 | 0.0360 | 0.8243 |
| **v2** asc+desc, random_fully_masked | 352.8 | 597.6 | 30.17 | 0.6760 | 0.0403 | 0.8087 |
| **v2** asc+desc, random_clouds | 311.7 | 513.8 | 30.69 | 0.6586 | 0.0393 | 0.8215 |

## 2. Métriques sur pixels reconstruits (Occluded)

### Masquage : Random Fully Masked

| Modèle | MAE | RMSE | PSNR | SSIM | SAM | R2 |
|:---|:---:|:---:|:---:|:---:|:---:|:---:|
| **ALL_SAR_120_epochs** (mix_closest) | 899.0 | 1108.3 | 22.54 | 0.4409 | 0.0886 | 0.6645 |
| asc+desc, random_clouds, DA | 998.2 | 1336.7 | 23.21 | 0.4328 | 0.2314 | 0.3926 |
| asc+desc, random_clouds | 983.2 | 1336.8 | 23.72 | 0.4275 | 0.2681 | 0.3750 |
| asc+desc, random_fully_masked, DA | 1055.5 | 1325.0 | 21.31 | 0.4019 | 0.1014 | 0.6231 |
| mix_closest, random_fully_masked | 1140.2 | 1412.7 | 21.09 | 0.4173 | 0.1036 | 0.6235 |
| mix_closest, random_fully_masked, DA | 1151.2 | 1440.3 | 21.14 | 0.4576 | 0.1010 | 0.6283 |
| coherence_only | 984.0 | 1314.7 | 22.04 | 0.3466 | 0.2360 | 0.4000 |
| **v2** mix_closest, random_fully_masked | 1085.4 | 1361.0 | 21.32 | 0.4587 | 0.0976 | 0.6275 |
| **v2** mix_closest, random_clouds | 900.9 | 1120.3 | 22.26 | 0.4566 | 0.0903 | 0.6500 |
| **v2** asc+desc, random_fully_masked | 1070.0 | 1343.3 | 21.33 | 0.4591 | 0.0980 | 0.6270 |
| **v2** asc+desc, random_clouds | 916.5 | 1128.8 | 22.04 | 0.4229 | 0.0965 | 0.6420 |

### Masquage : Consecutive Fully Masked

| Modèle | MAE | RMSE | PSNR | SSIM | SAM | R2 |
|:---|:---:|:---:|:---:|:---:|:---:|:---:|
| **ALL_SAR_120_epochs** (mix_closest) | 1081.9 | 1366.0 | 19.35 | 0.2190 | 0.1284 | 0.4716 |
| asc+desc, random_clouds, DA | 2296.9 | 2595.1 | 16.45 | 0.1560 | 0.6064 | 0.0721 |
| asc+desc, random_clouds | 2266.2 | 2570.1 | 17.09 | 0.1581 | 0.8594 | 0.0573 |
| asc+desc, random_fully_masked, DA | 1157.0 | 1512.9 | 19.39 | 0.2904 | 0.1122 | 0.5013 |
| mix_closest, random_fully_masked | 1305.4 | 1650.9 | 18.62 | 0.2937 | 0.1147 | 0.4997 |
| mix_closest, random_fully_masked, DA | 1437.6 | 1822.1 | 17.70 | 0.2879 | 0.1353 | 0.4253 |
| coherence_only | 2309.3 | 2608.6 | 15.12 | 0.1041 | 0.5966 | 0.1403 |
| **v2** mix_closest, random_fully_masked | 1195.6 | 1559.4 | 19.31 | 0.3089 | 0.1106 | 0.5010 |
| **v2** mix_closest, random_clouds | 1032.5 | 1335.7 | 19.74 | 0.2766 | 0.1151 | 0.4927 |
| **v2** asc+desc, random_fully_masked | 1192.1 | 1559.4 | 19.13 | 0.3149 | 0.1111 | 0.4921 |
| **v2** asc+desc, random_clouds | 1055.4 | 1341.4 | 19.59 | 0.2374 | 0.1191 | 0.4830 |

## 3. Occluded vs Observed (détail)

### Masquage : Random Fully Masked

| Modèle | Type | MAE | RMSE | PSNR | SSIM | SAM | R2 |
|:---|:---|:---:|:---:|:---:|:---:|:---:|:---:|
| **ALL_SAR_120_epochs** (mix_closest) | Occluded | 899.0 | 1108.3 | 22.54 | 0.4409 | 0.0886 | 0.6645 |
| **ALL_SAR_120_epochs** (mix_closest) | Observed | 69.2 | 99.0 | 40.19 | 0.8106 | 0.0288 | 0.9911 |
| asc+desc, random_clouds, DA | Occluded | 998.2 | 1336.7 | 23.21 | 0.4328 | 0.2314 | 0.3926 |
| asc+desc, random_clouds, DA | Observed | 64.3 | 93.8 | 40.71 | 0.8186 | 0.0271 | 0.9906 |
| asc+desc, random_clouds | Occluded | 983.2 | 1336.8 | 23.72 | 0.4275 | 0.2681 | 0.3750 |
| asc+desc, random_clouds | Observed | 56.1 | 82.1 | 41.83 | 0.8450 | 0.0234 | 0.9932 |
| asc+desc, random_fully_masked, DA | Occluded | 1055.5 | 1325.0 | 21.31 | 0.4019 | 0.1014 | 0.6231 |
| asc+desc, random_fully_masked, DA | Observed | 67.7 | 98.3 | 40.33 | 0.8276 | 0.0290 | 0.9891 |
| mix_closest, random_fully_masked | Occluded | 1140.2 | 1412.7 | 21.09 | 0.4173 | 0.1036 | 0.6235 |
| mix_closest, random_fully_masked | Observed | 71.1 | 102.9 | 39.94 | 0.8191 | 0.0300 | 0.9877 |
| mix_closest, random_fully_masked, DA | Occluded | 1151.2 | 1440.3 | 21.14 | 0.4576 | 0.1010 | 0.6283 |
| mix_closest, random_fully_masked, DA | Observed | 64.7 | 94.5 | 40.64 | 0.8339 | 0.0263 | 0.9912 |
| coherence_only | Occluded | 984.0 | 1314.7 | 22.04 | 0.3466 | 0.2360 | 0.4000 |
| coherence_only | Observed | 71.3 | 103.8 | 39.83 | 0.8044 | 0.0299 | 0.9883 |
| **v2** mix_closest, random_fully_masked | Occluded | 1085.4 | 1361.0 | 21.32 | 0.4587 | 0.0976 | 0.6275 |
| **v2** mix_closest, random_fully_masked | Observed | 73.3 | 106.3 | 39.63 | 0.8173 | 0.0311 | 0.9867 |
| **v2** mix_closest, random_clouds | Occluded | 900.9 | 1120.3 | 22.26 | 0.4566 | 0.0903 | 0.6500 |
| **v2** mix_closest, random_clouds | Observed | 59.7 | 85.5 | 41.51 | 0.8306 | 0.0249 | 0.9922 |
| **v2** asc+desc, random_fully_masked | Occluded | 1070.0 | 1343.3 | 21.33 | 0.4591 | 0.0980 | 0.6270 |
| **v2** asc+desc, random_fully_masked | Observed | 71.5 | 103.5 | 39.87 | 0.8123 | 0.0301 | 0.9875 |
| **v2** asc+desc, random_clouds | Occluded | 916.5 | 1128.8 | 22.04 | 0.4229 | 0.0965 | 0.6420 |
| **v2** asc+desc, random_clouds | Observed | 66.2 | 96.1 | 40.52 | 0.8095 | 0.0282 | 0.9894 |

### Masquage : Consecutive Fully Masked

| Modèle | Type | MAE | RMSE | PSNR | SSIM | SAM | R2 |
|:---|:---|:---:|:---:|:---:|:---:|:---:|:---:|
| **ALL_SAR_120_epochs** (mix_closest) | Occluded | 1081.9 | 1366.0 | 19.35 | 0.2190 | 0.1284 | 0.4716 |
| **ALL_SAR_120_epochs** (mix_closest) | Observed | 69.4 | 99.4 | 40.16 | 0.8098 | 0.0289 | 0.9910 |
| asc+desc, random_clouds, DA | Occluded | 2296.9 | 2595.1 | 16.45 | 0.1560 | 0.6064 | 0.0721 |
| asc+desc, random_clouds, DA | Observed | 64.5 | 94.2 | 40.67 | 0.8180 | 0.0271 | 0.9905 |
| asc+desc, random_clouds | Occluded | 2266.2 | 2570.1 | 17.09 | 0.1581 | 0.8594 | 0.0573 |
| asc+desc, random_clouds | Observed | 56.3 | 82.3 | 41.81 | 0.8445 | 0.0234 | 0.9932 |
| asc+desc, random_fully_masked, DA | Occluded | 1157.0 | 1512.9 | 19.39 | 0.2904 | 0.1122 | 0.5013 |
| asc+desc, random_fully_masked, DA | Observed | 67.9 | 98.5 | 40.31 | 0.8272 | 0.0290 | 0.9890 |
| mix_closest, random_fully_masked | Occluded | 1305.4 | 1650.9 | 18.62 | 0.2937 | 0.1147 | 0.4997 |
| mix_closest, random_fully_masked | Observed | 71.3 | 103.1 | 39.92 | 0.8185 | 0.0300 | 0.9877 |
| mix_closest, random_fully_masked, DA | Occluded | 1437.6 | 1822.1 | 17.70 | 0.2879 | 0.1353 | 0.4253 |
| mix_closest, random_fully_masked, DA | Observed | 65.1 | 94.9 | 40.60 | 0.8332 | 0.0264 | 0.9912 |
| coherence_only | Occluded | 2309.3 | 2608.6 | 15.12 | 0.1041 | 0.5966 | 0.1403 |
| coherence_only | Observed | 71.6 | 104.2 | 39.79 | 0.8038 | 0.0300 | 0.9883 |
| **v2** mix_closest, random_fully_masked | Occluded | 1195.6 | 1559.4 | 19.31 | 0.3089 | 0.1106 | 0.5010 |
| **v2** mix_closest, random_fully_masked | Observed | 73.7 | 106.7 | 39.60 | 0.8166 | 0.0311 | 0.9867 |
| **v2** mix_closest, random_clouds | Occluded | 1032.5 | 1335.7 | 19.74 | 0.2766 | 0.1151 | 0.4927 |
| **v2** mix_closest, random_clouds | Observed | 59.9 | 85.8 | 41.47 | 0.8300 | 0.0249 | 0.9922 |
| **v2** asc+desc, random_fully_masked | Occluded | 1192.1 | 1559.4 | 19.13 | 0.3149 | 0.1111 | 0.4921 |
| **v2** asc+desc, random_fully_masked | Observed | 71.9 | 104.1 | 39.82 | 0.8117 | 0.0302 | 0.9875 |
| **v2** asc+desc, random_clouds | Occluded | 1055.4 | 1341.4 | 19.59 | 0.2374 | 0.1191 | 0.4830 |
| **v2** asc+desc, random_clouds | Observed | 66.4 | 96.4 | 40.48 | 0.8088 | 0.0283 | 0.9893 |

## 4. Meilleurs modèles — Métriques Occluded

### Random Fully Masked

#### Classement (pixels occluded, score = rang moyen pondéré)

| # | Modèle | Score | MAE | RMSE | PSNR | SSIM | SAM | R2 |
|:---:|:---|:---:|:---:|:---:|:---:|:---:|:---:|:---:|
| 1 | **ALL_SAR_120_epochs** (mix_closest) | 1.67 | 899.0 | 1108.3 | 22.54 | 0.4409 | 0.0886 | 0.6645 |
| 2 | **v2** mix_closest, random_clouds | 2.44 | 900.9 | 1120.3 | 22.26 | 0.4566 | 0.0903 | 0.6500 |
| 3 | **v2** asc+desc, random_clouds | 3.89 | 916.5 | 1128.8 | 22.04 | 0.4229 | 0.0965 | 0.6420 |
| 4 | **v2** asc+desc, random_fully_masked | 6.33 | 1070.0 | 1343.3 | 21.33 | 0.4591 | 0.0980 | 0.6270 |
| 5 | **v2** mix_closest, random_fully_masked | 6.67 | 1085.4 | 1361.0 | 21.32 | 0.4587 | 0.0976 | 0.6275 |

#### Détail par bande du meilleur modèle : **ALL_SAR_120_epochs** (mix_closest)

| Métrique | B2 | B3 | B4 | B5 | B6 | B7 | B8 | B8A | B11 | B12 |
|:---|:---:|:---:|:---:|:---:|:---:|:---:|:---:|:---:|:---:|:---:|
| MAE | 517.3 | 595.0 | 634.3 | 743.6 | 1050.9 | 1186.0 | 1252.1 | 1248.4 | 988.8 | 774.0 |
| RMSE | 635.4 | 718.7 | 771.0 | 880.8 | 1218.5 | 1373.3 | 1445.0 | 1440.2 | 1134.7 | 904.3 |
| PSNR | 28.25 | 27.17 | 25.48 | 25.04 | 22.04 | 20.80 | 20.23 | 20.54 | 22.50 | 23.82 |
| SSIM | 0.3287 | 0.3831 | 0.3937 | 0.4551 | 0.4868 | 0.4919 | 0.4484 | 0.5003 | 0.4670 | 0.4536 |
| R2 | 0.3984 | 0.4447 | 0.4628 | 0.4506 | 0.4513 | 0.4715 | 0.4661 | 0.4844 | 0.4648 | 0.4900 |

### Consecutive Fully Masked

#### Classement (pixels occluded, score = rang moyen pondéré)

| # | Modèle | Score | MAE | RMSE | PSNR | SSIM | SAM | R2 |
|:---:|:---|:---:|:---:|:---:|:---:|:---:|:---:|:---:|
| 1 | **v2** mix_closest, random_clouds | 2.67 | 1032.5 | 1335.7 | 19.74 | 0.2766 | 0.1151 | 0.4927 |
| 2 | asc+desc, random_fully_masked, DA | 3.11 | 1157.0 | 1512.9 | 19.39 | 0.2904 | 0.1122 | 0.5013 |
| 3 | **v2** mix_closest, random_fully_masked | 3.78 | 1195.6 | 1559.4 | 19.31 | 0.3089 | 0.1106 | 0.5010 |
| 4 | **v2** asc+desc, random_clouds | 3.89 | 1055.4 | 1341.4 | 19.59 | 0.2374 | 0.1191 | 0.4830 |
| 5 | **v2** asc+desc, random_fully_masked | 4.56 | 1192.1 | 1559.4 | 19.13 | 0.3149 | 0.1111 | 0.4921 |

#### Détail par bande du meilleur modèle : **v2** mix_closest, random_clouds

| Métrique | B2 | B3 | B4 | B5 | B6 | B7 | B8 | B8A | B11 | B12 |
|:---|:---:|:---:|:---:|:---:|:---:|:---:|:---:|:---:|:---:|:---:|
| MAE | 630.1 | 700.5 | 737.2 | 851.7 | 1223.0 | 1368.7 | 1427.0 | 1427.3 | 1089.5 | 869.8 |
| RMSE | 911.9 | 961.9 | 995.1 | 1103.7 | 1466.8 | 1621.0 | 1683.3 | 1683.1 | 1270.1 | 1026.6 |
| PSNR | 23.62 | 23.15 | 22.31 | 21.72 | 19.17 | 18.20 | 17.82 | 17.93 | 20.32 | 21.79 |
| SSIM | 0.2018 | 0.2304 | 0.2403 | 0.2770 | 0.2991 | 0.3060 | 0.2901 | 0.3136 | 0.3112 | 0.2961 |
| R2 | 0.1798 | 0.2209 | 0.2615 | 0.2437 | 0.2127 | 0.2307 | 0.2398 | 0.2391 | 0.2838 | 0.3059 |

## 5. Disponibilité des résultats

| Modèle | Random Fully Masked | Consecutive Fully Masked |
|:---|:---:|:---:|
| **ALL_SAR_120_epochs** (mix_closest) | ✅ | ✅ |
| asc+desc, random_clouds, DA | ✅ | ✅ |
| asc+desc, random_clouds | ✅ | ✅ |
| asc+desc, random_fully_masked, DA | ✅ | ✅ |
| mix_closest, random_fully_masked | ✅ | ✅ |
| mix_closest, random_fully_masked, DA | ✅ | ✅ |
| coherence_only | ✅ | ✅ |
| **v2** mix_closest, random_fully_masked | ✅ | ✅ |
| **v2** mix_closest, random_clouds | ✅ | ✅ |
| **v2** asc+desc, random_fully_masked | ✅ | ✅ |
| **v2** asc+desc, random_clouds | ✅ | ✅ |


# Rapport de Métriques — Cloud Reconstruction U-TILISE

**Source des résultats :** logs SLURM dans `/mnt/stores/store_dai/tmp/speillet/logs`

## 1. Récapitulatif Global

### Masquage : Random Fully Masked

| Modèle | MAE | RMSE | PSNR | SSIM | SAM | R2 |
|:---|:---:|:---:|:---:|:---:|:---:|:---:|
| **ALL_SAR_120_epochs** (mix_closest) | 305.3 | 466.4 | 31.89 | 0.7129 | 0.0385 | 0.8487 |
| asc+desc, random_clouds, DA | 246.3 | 513.4 | 30.48 | 0.7515 | 0.0664 | 0.7896 |
| asc+desc, random_clouds | 239.6 | 512.5 | 30.83 | 0.7711 | 0.0695 | 0.7869 |
| asc+desc, random_fully_masked, DA | 332.9 | 534.7 | 31.30 | 0.7202 | 0.0404 | 0.8343 |
| mix_closest, random_fully_masked | 375.2 | 585.6 | 31.01 | 0.7167 | 0.0416 | 0.8306 |
| mix_closest, random_fully_masked, DA | 389.5 | 610.2 | 31.21 | 0.7357 | 0.0382 | 0.8353 |
| coherence_only | 243.9 | 504.7 | 30.23 | 0.7048 | 0.0696 | 0.7935 |
| **v2** mix_closest, random_fully_masked | 341.8 | 551.4 | 30.99 | 0.7234 | 0.0417 | 0.8305 |
| **v2** mix_closest, random_clouds | 107.4 | 204.7 | 35.69 | 0.7326 | 0.0353 | 0.9527 |
| **v2** asc+desc, random_fully_masked | 343.5 | 548.8 | 31.08 | 0.7200 | 0.0408 | 0.8313 |
| **v2** asc+desc, random_clouds | 303.6 | 469.4 | 31.73 | 0.7091 | 0.0392 | 0.8433 |
| **v3** mix_closest, random_clouds | 122.6 | 226.6 | 34.66 | 0.7039 | 0.0420 | 0.9432 |

### Masquage : Consecutive Fully Masked

| Modèle | MAE | RMSE | PSNR | SSIM | SAM | R2 |
|:---|:---:|:---:|:---:|:---:|:---:|:---:|
| **ALL_SAR_120_epochs** (mix_closest) | 320.4 | 524.9 | 30.39 | 0.6548 | 0.0411 | 0.8197 |
| asc+desc, random_clouds, DA | 364.2 | 819.4 | 26.51 | 0.6881 | 0.0987 | 0.6286 |
| asc+desc, random_clouds | 355.0 | 810.5 | 26.95 | 0.7086 | 0.1284 | 0.6326 |
| asc+desc, random_fully_masked, DA | 339.4 | 576.2 | 30.56 | 0.6832 | 0.0395 | 0.8166 |
| mix_closest, random_fully_masked | 388.6 | 640.2 | 29.93 | 0.6772 | 0.0406 | 0.8054 |
| mix_closest, random_fully_masked, DA | 421.1 | 709.6 | 29.47 | 0.6861 | 0.0399 | 0.7864 |
| coherence_only | 371.0 | 826.0 | 26.16 | 0.6488 | 0.1002 | 0.6264 |
| **v2** mix_closest, random_fully_masked | 350.0 | 595.5 | 30.21 | 0.6783 | 0.0412 | 0.8110 |
| **v2** mix_closest, random_clouds | 117.0 | 262.6 | 34.26 | 0.6824 | 0.0360 | 0.9236 |
| **v2** asc+desc, random_fully_masked | 352.8 | 597.6 | 30.17 | 0.6760 | 0.0403 | 0.8087 |
| **v2** asc+desc, random_clouds | 311.7 | 513.8 | 30.69 | 0.6586 | 0.0393 | 0.8215 |
| **v3** mix_closest, random_clouds | 135.8 | 286.3 | 33.27 | 0.6651 | 0.0419 | 0.9132 |

## 2. Métriques sur pixels reconstruits (Occluded)

### Masquage : Random Fully Masked

| Modèle | MAE | RMSE | PSNR | SSIM | SAM | R2 |
|:---|:---:|:---:|:---:|:---:|:---:|:---:|
| **ALL_SAR_120_epochs** (mix_closest) | 899.0 | 1108.3 | 22.54 | 0.4409 | 0.0886 | 0.6645 |
| asc+desc, random_clouds, DA | 998.2 | 1336.7 | 23.21 | 0.4328 | 0.2314 | 0.3926 |
| asc+desc, random_clouds | 983.2 | 1336.8 | 23.72 | 0.4275 | 0.2681 | 0.3750 |
| asc+desc, random_fully_masked, DA | 1055.5 | 1325.0 | 21.31 | 0.4019 | 0.1014 | 0.6231 |
| mix_closest, random_fully_masked | 1140.2 | 1412.7 | 21.09 | 0.4173 | 0.1036 | 0.6235 |
| mix_closest, random_fully_masked, DA | 1151.2 | 1440.3 | 21.14 | 0.4576 | 0.1010 | 0.6283 |
| coherence_only | 984.0 | 1314.7 | 22.04 | 0.3466 | 0.2360 | 0.4000 |
| **v2** mix_closest, random_fully_masked | 1085.4 | 1361.0 | 21.32 | 0.4587 | 0.0976 | 0.6275 |
| **v2** mix_closest, random_clouds | 360.5 | 538.3 | 26.36 | 0.4566 | 0.0903 | 0.7770 |
| **v2** asc+desc, random_fully_masked | 1070.0 | 1343.3 | 21.33 | 0.4591 | 0.0980 | 0.6270 |
| **v2** asc+desc, random_clouds | 916.5 | 1128.8 | 22.04 | 0.4229 | 0.0965 | 0.6420 |
| **v3** mix_closest, random_clouds | 406.1 | 585.0 | 25.54 | 0.3765 | 0.1065 | 0.7422 |

### Masquage : Consecutive Fully Masked

| Modèle | MAE | RMSE | PSNR | SSIM | SAM | R2 |
|:---|:---:|:---:|:---:|:---:|:---:|:---:|
| **ALL_SAR_120_epochs** (mix_closest) | 1081.9 | 1366.0 | 19.35 | 0.2190 | 0.1284 | 0.4716 |
| asc+desc, random_clouds, DA | 2296.9 | 2595.1 | 16.45 | 0.1560 | 0.6064 | 0.0721 |
| asc+desc, random_clouds | 2266.2 | 2570.1 | 17.09 | 0.1581 | 0.8594 | 0.0573 |
| asc+desc, random_fully_masked, DA | 1157.0 | 1512.9 | 19.39 | 0.2904 | 0.1122 | 0.5013 |
| mix_closest, random_fully_masked | 1305.4 | 1650.9 | 18.62 | 0.2937 | 0.1147 | 0.4997 |
| mix_closest, random_fully_masked, DA | 1437.6 | 1822.1 | 17.70 | 0.2879 | 0.1353 | 0.4253 |
| coherence_only | 2309.3 | 2608.6 | 15.12 | 0.1041 | 0.5966 | 0.1403 |
| **v2** mix_closest, random_fully_masked | 1195.6 | 1559.4 | 19.31 | 0.3089 | 0.1106 | 0.5010 |
| **v2** mix_closest, random_clouds | 542.1 | 847.0 | 22.71 | 0.2766 | 0.1151 | 0.5809 |
| **v2** asc+desc, random_fully_masked | 1192.1 | 1559.4 | 19.13 | 0.3149 | 0.1111 | 0.4921 |
| **v2** asc+desc, random_clouds | 1055.4 | 1341.4 | 19.59 | 0.2374 | 0.1191 | 0.4830 |
| **v3** mix_closest, random_clouds | 626.4 | 907.3 | 21.91 | 0.2504 | 0.1291 | 0.5601 |

## 3. Occluded vs Observed (détail)

### Masquage : Random Fully Masked

| Modèle | Type | MAE | RMSE | PSNR | SSIM | SAM | R2 |
|:---|:---|:---:|:---:|:---:|:---:|:---:|:---:|
| **ALL_SAR_120_epochs** (mix_closest) | Occluded | 899.0 | 1108.3 | 22.54 | 0.4409 | 0.0886 | 0.6645 |
| **ALL_SAR_120_epochs** (mix_closest) | Observed | 69.2 | 99.0 | 40.19 | 0.8106 | 0.0288 | 0.9911 |
| asc+desc, random_clouds, DA | Occluded | 998.2 | 1336.7 | 23.21 | 0.4328 | 0.2314 | 0.3926 |
| asc+desc, random_clouds, DA | Observed | 64.3 | 93.8 | 40.71 | 0.8186 | 0.0271 | 0.9906 |
| asc+desc, random_clouds | Occluded | 983.2 | 1336.8 | 23.72 | 0.4275 | 0.2681 | 0.3750 |
| asc+desc, random_clouds | Observed | 56.1 | 82.1 | 41.83 | 0.8450 | 0.0234 | 0.9932 |
| asc+desc, random_fully_masked, DA | Occluded | 1055.5 | 1325.0 | 21.31 | 0.4019 | 0.1014 | 0.6231 |
| asc+desc, random_fully_masked, DA | Observed | 67.7 | 98.3 | 40.33 | 0.8276 | 0.0290 | 0.9891 |
| mix_closest, random_fully_masked | Occluded | 1140.2 | 1412.7 | 21.09 | 0.4173 | 0.1036 | 0.6235 |
| mix_closest, random_fully_masked | Observed | 71.1 | 102.9 | 39.94 | 0.8191 | 0.0300 | 0.9877 |
| mix_closest, random_fully_masked, DA | Occluded | 1151.2 | 1440.3 | 21.14 | 0.4576 | 0.1010 | 0.6283 |
| mix_closest, random_fully_masked, DA | Observed | 64.7 | 94.5 | 40.64 | 0.8339 | 0.0263 | 0.9912 |
| coherence_only | Occluded | 984.0 | 1314.7 | 22.04 | 0.3466 | 0.2360 | 0.4000 |
| coherence_only | Observed | 71.3 | 103.8 | 39.83 | 0.8044 | 0.0299 | 0.9883 |
| **v2** mix_closest, random_fully_masked | Occluded | 1085.4 | 1361.0 | 21.32 | 0.4587 | 0.0976 | 0.6275 |
| **v2** mix_closest, random_fully_masked | Observed | 73.3 | 106.3 | 39.63 | 0.8173 | 0.0311 | 0.9867 |
| **v2** mix_closest, random_clouds | Occluded | 360.5 | 538.3 | 26.36 | 0.4566 | 0.0903 | 0.7770 |
| **v2** mix_closest, random_clouds | Observed | 59.7 | 85.5 | 41.51 | 0.8306 | 0.0249 | 0.9922 |
| **v2** asc+desc, random_fully_masked | Occluded | 1070.0 | 1343.3 | 21.33 | 0.4591 | 0.0980 | 0.6270 |
| **v2** asc+desc, random_fully_masked | Observed | 71.5 | 103.5 | 39.87 | 0.8123 | 0.0301 | 0.9875 |
| **v2** asc+desc, random_clouds | Occluded | 916.5 | 1128.8 | 22.04 | 0.4229 | 0.0965 | 0.6420 |
| **v2** asc+desc, random_clouds | Observed | 66.2 | 96.1 | 40.52 | 0.8095 | 0.0282 | 0.9894 |
| **v3** mix_closest, random_clouds | Occluded | 406.1 | 585.0 | 25.54 | 0.3765 | 0.1065 | 0.7422 |
| **v3** mix_closest, random_clouds | Observed | 68.9 | 100.0 | 40.19 | 0.8142 | 0.0298 | 0.9883 |

### Masquage : Consecutive Fully Masked

| Modèle | Type | MAE | RMSE | PSNR | SSIM | SAM | R2 |
|:---|:---|:---:|:---:|:---:|:---:|:---:|:---:|
| **ALL_SAR_120_epochs** (mix_closest) | Occluded | 1081.9 | 1366.0 | 19.35 | 0.2190 | 0.1284 | 0.4716 |
| **ALL_SAR_120_epochs** (mix_closest) | Observed | 69.4 | 99.4 | 40.16 | 0.8098 | 0.0289 | 0.9910 |
| asc+desc, random_clouds, DA | Occluded | 2296.9 | 2595.1 | 16.45 | 0.1560 | 0.6064 | 0.0721 |
| asc+desc, random_clouds, DA | Observed | 64.5 | 94.2 | 40.67 | 0.8180 | 0.0271 | 0.9905 |
| asc+desc, random_clouds | Occluded | 2266.2 | 2570.1 | 17.09 | 0.1581 | 0.8594 | 0.0573 |
| asc+desc, random_clouds | Observed | 56.3 | 82.3 | 41.81 | 0.8445 | 0.0234 | 0.9932 |
| asc+desc, random_fully_masked, DA | Occluded | 1157.0 | 1512.9 | 19.39 | 0.2904 | 0.1122 | 0.5013 |
| asc+desc, random_fully_masked, DA | Observed | 67.9 | 98.5 | 40.31 | 0.8272 | 0.0290 | 0.9890 |
| mix_closest, random_fully_masked | Occluded | 1305.4 | 1650.9 | 18.62 | 0.2937 | 0.1147 | 0.4997 |
| mix_closest, random_fully_masked | Observed | 71.3 | 103.1 | 39.92 | 0.8185 | 0.0300 | 0.9877 |
| mix_closest, random_fully_masked, DA | Occluded | 1437.6 | 1822.1 | 17.70 | 0.2879 | 0.1353 | 0.4253 |
| mix_closest, random_fully_masked, DA | Observed | 65.1 | 94.9 | 40.60 | 0.8332 | 0.0264 | 0.9912 |
| coherence_only | Occluded | 2309.3 | 2608.6 | 15.12 | 0.1041 | 0.5966 | 0.1403 |
| coherence_only | Observed | 71.6 | 104.2 | 39.79 | 0.8038 | 0.0300 | 0.9883 |
| **v2** mix_closest, random_fully_masked | Occluded | 1195.6 | 1559.4 | 19.31 | 0.3089 | 0.1106 | 0.5010 |
| **v2** mix_closest, random_fully_masked | Observed | 73.7 | 106.7 | 39.60 | 0.8166 | 0.0311 | 0.9867 |
| **v2** mix_closest, random_clouds | Occluded | 542.1 | 847.0 | 22.71 | 0.2766 | 0.1151 | 0.5809 |
| **v2** mix_closest, random_clouds | Observed | 59.9 | 85.8 | 41.47 | 0.8300 | 0.0249 | 0.9922 |
| **v2** asc+desc, random_fully_masked | Occluded | 1192.1 | 1559.4 | 19.13 | 0.3149 | 0.1111 | 0.4921 |
| **v2** asc+desc, random_fully_masked | Observed | 71.9 | 104.1 | 39.82 | 0.8117 | 0.0302 | 0.9875 |
| **v2** asc+desc, random_clouds | Occluded | 1055.4 | 1341.4 | 19.59 | 0.2374 | 0.1191 | 0.4830 |
| **v2** asc+desc, random_clouds | Observed | 66.4 | 96.4 | 40.48 | 0.8088 | 0.0283 | 0.9893 |
| **v3** mix_closest, random_clouds | Occluded | 626.4 | 907.3 | 21.91 | 0.2504 | 0.1291 | 0.5601 |
| **v3** mix_closest, random_clouds | Observed | 69.2 | 100.4 | 40.15 | 0.8135 | 0.0298 | 0.9883 |

## 4. Meilleurs modèles — Métriques Occluded

### Random Fully Masked

#### Classement (pixels occluded, score = rang moyen pondéré)

| # | Modèle | Score | MAE | RMSE | PSNR | SSIM | SAM | R2 |
|:---:|:---|:---:|:---:|:---:|:---:|:---:|:---:|:---:|
| 1 | **v2** mix_closest, random_clouds | 1.44 | 360.5 | 538.3 | 26.36 | 0.4566 | 0.0903 | 0.7770 |
| 2 | **ALL_SAR_120_epochs** (mix_closest) | 3.22 | 899.0 | 1108.3 | 22.54 | 0.4409 | 0.0886 | 0.6645 |
| 3 | **v3** mix_closest, random_clouds | 3.78 | 406.1 | 585.0 | 25.54 | 0.3765 | 0.1065 | 0.7422 |
| 4 | **v2** asc+desc, random_clouds | 4.67 | 916.5 | 1128.8 | 22.04 | 0.4229 | 0.0965 | 0.6420 |
| 5 | **v2** asc+desc, random_fully_masked | 7.11 | 1070.0 | 1343.3 | 21.33 | 0.4591 | 0.0980 | 0.6270 |

#### Détail par bande du meilleur modèle : **v2** mix_closest, random_clouds

| Métrique | B2 | B3 | B4 | B5 | B6 | B7 | B8 | B8A | B11 | B12 |
|:---|:---:|:---:|:---:|:---:|:---:|:---:|:---:|:---:|:---:|:---:|
| MAE | 209.0 | 221.9 | 292.8 | 284.3 | 400.8 | 478.0 | 509.1 | 480.7 | 375.2 | 353.0 |
| RMSE | 335.6 | 349.1 | 447.9 | 424.4 | 550.2 | 644.7 | 679.5 | 645.2 | 512.2 | 485.9 |
| PSNR | 32.55 | 31.66 | 28.77 | 29.24 | 26.16 | 24.65 | 24.11 | 24.60 | 26.52 | 27.09 |
| SSIM | 0.3661 | 0.4102 | 0.4067 | 0.4740 | 0.4865 | 0.4885 | 0.4686 | 0.4997 | 0.4937 | 0.4721 |
| R2 | 0.4490 | 0.5062 | 0.4971 | 0.5012 | 0.5062 | 0.5218 | 0.5210 | 0.5419 | 0.5152 | 0.5252 |

### Consecutive Fully Masked

#### Classement (pixels occluded, score = rang moyen pondéré)

| # | Modèle | Score | MAE | RMSE | PSNR | SSIM | SAM | R2 |
|:---:|:---|:---:|:---:|:---:|:---:|:---:|:---:|:---:|
| 1 | **v2** mix_closest, random_clouds | 2.00 | 542.1 | 847.0 | 22.71 | 0.2766 | 0.1151 | 0.5809 |
| 2 | **v3** mix_closest, random_clouds | 3.22 | 626.4 | 907.3 | 21.91 | 0.2504 | 0.1291 | 0.5601 |
| 3 | asc+desc, random_fully_masked, DA | 4.11 | 1157.0 | 1512.9 | 19.39 | 0.2904 | 0.1122 | 0.5013 |
| 4 | **v2** mix_closest, random_fully_masked | 4.78 | 1195.6 | 1559.4 | 19.31 | 0.3089 | 0.1106 | 0.5010 |
| 5 | **v2** asc+desc, random_clouds | 4.78 | 1055.4 | 1341.4 | 19.59 | 0.2374 | 0.1191 | 0.4830 |

#### Détail par bande du meilleur modèle : **v2** mix_closest, random_clouds

| Métrique | B2 | B3 | B4 | B5 | B6 | B7 | B8 | B8A | B11 | B12 |
|:---|:---:|:---:|:---:|:---:|:---:|:---:|:---:|:---:|:---:|:---:|
| MAE | 387.5 | 395.8 | 456.7 | 463.1 | 614.7 | 695.0 | 727.2 | 709.5 | 516.4 | 454.6 |
| RMSE | 741.8 | 717.1 | 771.7 | 762.6 | 874.1 | 953.4 | 987.4 | 965.6 | 691.7 | 614.9 |
| PSNR | 26.04 | 25.89 | 24.64 | 24.59 | 22.42 | 21.42 | 21.04 | 21.23 | 23.70 | 24.76 |
| SSIM | 0.2018 | 0.2304 | 0.2403 | 0.2770 | 0.2991 | 0.3060 | 0.2901 | 0.3136 | 0.3112 | 0.2961 |
| R2 | 0.2062 | 0.2555 | 0.3012 | 0.2817 | 0.2395 | 0.2634 | 0.2745 | 0.2748 | 0.3254 | 0.3512 |

## 5. Disponibilité des résultats

| Modèle | Random Fully Masked | Consecutive Fully Masked |
|:---|:---:|:---:|
| **ALL_SAR_120_epochs** (mix_closest) | ✅ | ✅ |
| asc+desc, random_clouds, DA | ✅ | ✅ |
| asc+desc, random_clouds | ✅ | ✅ |
| asc+desc, random_fully_masked, DA | ✅ | ✅ |
| mix_closest, random_fully_masked | ✅ | ✅ |
| mix_closest, random_fully_masked, DA | ✅ | ✅ |
| coherence_only | ✅ | ✅ |
| **v2** mix_closest, random_fully_masked | ✅ | ✅ |
| **v2** mix_closest, random_clouds | ✅ | ✅ |
| **v2** asc+desc, random_fully_masked | ✅ | ✅ |
| **v2** asc+desc, random_clouds | ✅ | ✅ |
| **v3** mix_closest, random_clouds | ✅ | ✅ |


# Rapport de Métriques — Cloud Reconstruction U-TILISE

**Source des résultats :** logs SLURM dans `/mnt/stores/store_dai/tmp/speillet/logs` + JSON dans `/mnt/DATA_10T/data_rpg/outputs/U-TILISE/metrics`

## 1. Récapitulatif Global

### Masquage : Random Fully Masked

| Modèle | MAE | RMSE | PSNR | SSIM | SAM | R2 |
|:---|:---:|:---:|:---:|:---:|:---:|:---:|
| **ALL_SAR_120_epochs** (mix_closest) | 305.3 | 466.4 | 31.89 | 0.7129 | 0.0385 | 0.8487 |
| asc+desc, random_clouds, DA | 246.3 | 513.4 | 30.48 | 0.7515 | 0.0664 | 0.7896 |
| asc+desc, random_clouds | 239.6 | 512.5 | 30.83 | 0.7711 | 0.0695 | 0.7869 |
| asc+desc, random_fully_masked, DA | 332.9 | 534.7 | 31.30 | 0.7202 | 0.0404 | 0.8343 |
| mix_closest, random_fully_masked | 375.2 | 585.6 | 31.01 | 0.7167 | 0.0416 | 0.8306 |
| mix_closest, random_fully_masked, DA | 389.5 | 610.2 | 31.21 | 0.7357 | 0.0382 | 0.8353 |
| coherence_only | 243.9 | 504.7 | 30.23 | 0.7048 | 0.0696 | 0.7935 |
| **v2** mix_closest, random_fully_masked | 341.8 | 551.4 | 30.99 | 0.7234 | 0.0417 | 0.8305 |
| **v2** mix_closest, random_clouds | 107.4 | 204.7 | 35.69 | 0.7326 | 0.0353 | 0.9527 |
| **v2** asc+desc, random_fully_masked | 343.5 | 548.8 | 31.08 | 0.7200 | 0.0408 | 0.8313 |
| **v2** asc+desc, random_clouds | 303.6 | 469.4 | 31.73 | 0.7091 | 0.0392 | 0.8433 |
| **v3** mix_closest, random_clouds | 122.6 | 226.6 | 34.66 | 0.7039 | 0.0420 | 0.9432 |

### Masquage : Consecutive Fully Masked

| Modèle | MAE | RMSE | PSNR | SSIM | SAM | R2 |
|:---|:---:|:---:|:---:|:---:|:---:|:---:|
| **ALL_SAR_120_epochs** (mix_closest) | 320.4 | 524.9 | 30.39 | 0.6548 | 0.0411 | 0.8197 |
| asc+desc, random_clouds, DA | 364.2 | 819.4 | 26.51 | 0.6881 | 0.0987 | 0.6286 |
| asc+desc, random_clouds | 355.0 | 810.5 | 26.95 | 0.7086 | 0.1284 | 0.6326 |
| asc+desc, random_fully_masked, DA | 339.4 | 576.2 | 30.56 | 0.6832 | 0.0395 | 0.8166 |
| mix_closest, random_fully_masked | 388.6 | 640.2 | 29.93 | 0.6772 | 0.0406 | 0.8054 |
| mix_closest, random_fully_masked, DA | 421.1 | 709.6 | 29.47 | 0.6861 | 0.0399 | 0.7864 |
| coherence_only | 371.0 | 826.0 | 26.16 | 0.6488 | 0.1002 | 0.6264 |
| **v2** mix_closest, random_fully_masked | 350.0 | 595.5 | 30.21 | 0.6783 | 0.0412 | 0.8110 |
| **v2** mix_closest, random_clouds | 117.0 | 262.6 | 34.26 | 0.6824 | 0.0360 | 0.9236 |
| **v2** asc+desc, random_fully_masked | 352.8 | 597.6 | 30.17 | 0.6760 | 0.0403 | 0.8087 |
| **v2** asc+desc, random_clouds | 311.7 | 513.8 | 30.69 | 0.6586 | 0.0393 | 0.8215 |
| **v3** mix_closest, random_clouds | 135.8 | 286.3 | 33.27 | 0.6651 | 0.0419 | 0.9132 |

## 2. Métriques sur pixels reconstruits (Occluded)

### Masquage : Random Fully Masked

| Modèle | MAE | RMSE | PSNR | SSIM | SAM | R2 |
|:---|:---:|:---:|:---:|:---:|:---:|:---:|
| **ALL_SAR_120_epochs** (mix_closest) | 899.0 | 1108.3 | 22.54 | 0.4409 | 0.0886 | 0.6645 |
| asc+desc, random_clouds, DA | 998.2 | 1336.7 | 23.21 | 0.4328 | 0.2314 | 0.3926 |
| asc+desc, random_clouds | 983.2 | 1336.8 | 23.72 | 0.4275 | 0.2681 | 0.3750 |
| asc+desc, random_fully_masked, DA | 1055.5 | 1325.0 | 21.31 | 0.4019 | 0.1014 | 0.6231 |
| mix_closest, random_fully_masked | 1140.2 | 1412.7 | 21.09 | 0.4173 | 0.1036 | 0.6235 |
| mix_closest, random_fully_masked, DA | 1151.2 | 1440.3 | 21.14 | 0.4576 | 0.1010 | 0.6283 |
| coherence_only | 984.0 | 1314.7 | 22.04 | 0.3466 | 0.2360 | 0.4000 |
| **v2** mix_closest, random_fully_masked | 1085.4 | 1361.0 | 21.32 | 0.4587 | 0.0976 | 0.6275 |
| **v2** mix_closest, random_clouds | 360.5 | 538.3 | 26.36 | 0.4566 | 0.0903 | 0.7770 |
| **v2** asc+desc, random_fully_masked | 1070.0 | 1343.3 | 21.33 | 0.4591 | 0.0980 | 0.6270 |
| **v2** asc+desc, random_clouds | 916.5 | 1128.8 | 22.04 | 0.4229 | 0.0965 | 0.6420 |
| **v3** mix_closest, random_clouds | 406.1 | 585.0 | 25.54 | 0.3765 | 0.1065 | 0.7422 |

### Masquage : Consecutive Fully Masked

| Modèle | MAE | RMSE | PSNR | SSIM | SAM | R2 |
|:---|:---:|:---:|:---:|:---:|:---:|:---:|
| **ALL_SAR_120_epochs** (mix_closest) | 1081.9 | 1366.0 | 19.35 | 0.2190 | 0.1284 | 0.4716 |
| asc+desc, random_clouds, DA | 2296.9 | 2595.1 | 16.45 | 0.1560 | 0.6064 | 0.0721 |
| asc+desc, random_clouds | 2266.2 | 2570.1 | 17.09 | 0.1581 | 0.8594 | 0.0573 |
| asc+desc, random_fully_masked, DA | 1157.0 | 1512.9 | 19.39 | 0.2904 | 0.1122 | 0.5013 |
| mix_closest, random_fully_masked | 1305.4 | 1650.9 | 18.62 | 0.2937 | 0.1147 | 0.4997 |
| mix_closest, random_fully_masked, DA | 1437.6 | 1822.1 | 17.70 | 0.2879 | 0.1353 | 0.4253 |
| coherence_only | 2309.3 | 2608.6 | 15.12 | 0.1041 | 0.5966 | 0.1403 |
| **v2** mix_closest, random_fully_masked | 1195.6 | 1559.4 | 19.31 | 0.3089 | 0.1106 | 0.5010 |
| **v2** mix_closest, random_clouds | 542.1 | 847.0 | 22.71 | 0.2766 | 0.1151 | 0.5809 |
| **v2** asc+desc, random_fully_masked | 1192.1 | 1559.4 | 19.13 | 0.3149 | 0.1111 | 0.4921 |
| **v2** asc+desc, random_clouds | 1055.4 | 1341.4 | 19.59 | 0.2374 | 0.1191 | 0.4830 |
| **v3** mix_closest, random_clouds | 626.4 | 907.3 | 21.91 | 0.2504 | 0.1291 | 0.5601 |

## 3. Occluded vs Observed (détail)

### Masquage : Random Fully Masked

| Modèle | Type | MAE | RMSE | PSNR | SSIM | SAM | R2 |
|:---|:---|:---:|:---:|:---:|:---:|:---:|:---:|
| **ALL_SAR_120_epochs** (mix_closest) | Occluded | 899.0 | 1108.3 | 22.54 | 0.4409 | 0.0886 | 0.6645 |
| **ALL_SAR_120_epochs** (mix_closest) | Observed | 69.2 | 99.0 | 40.19 | 0.8106 | 0.0288 | 0.9911 |
| asc+desc, random_clouds, DA | Occluded | 998.2 | 1336.7 | 23.21 | 0.4328 | 0.2314 | 0.3926 |
| asc+desc, random_clouds, DA | Observed | 64.3 | 93.8 | 40.71 | 0.8186 | 0.0271 | 0.9906 |
| asc+desc, random_clouds | Occluded | 983.2 | 1336.8 | 23.72 | 0.4275 | 0.2681 | 0.3750 |
| asc+desc, random_clouds | Observed | 56.1 | 82.1 | 41.83 | 0.8450 | 0.0234 | 0.9932 |
| asc+desc, random_fully_masked, DA | Occluded | 1055.5 | 1325.0 | 21.31 | 0.4019 | 0.1014 | 0.6231 |
| asc+desc, random_fully_masked, DA | Observed | 67.7 | 98.3 | 40.33 | 0.8276 | 0.0290 | 0.9891 |
| mix_closest, random_fully_masked | Occluded | 1140.2 | 1412.7 | 21.09 | 0.4173 | 0.1036 | 0.6235 |
| mix_closest, random_fully_masked | Observed | 71.1 | 102.9 | 39.94 | 0.8191 | 0.0300 | 0.9877 |
| mix_closest, random_fully_masked, DA | Occluded | 1151.2 | 1440.3 | 21.14 | 0.4576 | 0.1010 | 0.6283 |
| mix_closest, random_fully_masked, DA | Observed | 64.7 | 94.5 | 40.64 | 0.8339 | 0.0263 | 0.9912 |
| coherence_only | Occluded | 984.0 | 1314.7 | 22.04 | 0.3466 | 0.2360 | 0.4000 |
| coherence_only | Observed | 71.3 | 103.8 | 39.83 | 0.8044 | 0.0299 | 0.9883 |
| **v2** mix_closest, random_fully_masked | Occluded | 1085.4 | 1361.0 | 21.32 | 0.4587 | 0.0976 | 0.6275 |
| **v2** mix_closest, random_fully_masked | Observed | 73.3 | 106.3 | 39.63 | 0.8173 | 0.0311 | 0.9867 |
| **v2** mix_closest, random_clouds | Occluded | 360.5 | 538.3 | 26.36 | 0.4566 | 0.0903 | 0.7770 |
| **v2** mix_closest, random_clouds | Observed | 59.7 | 85.5 | 41.51 | 0.8306 | 0.0249 | 0.9922 |
| **v2** asc+desc, random_fully_masked | Occluded | 1070.0 | 1343.3 | 21.33 | 0.4591 | 0.0980 | 0.6270 |
| **v2** asc+desc, random_fully_masked | Observed | 71.5 | 103.5 | 39.87 | 0.8123 | 0.0301 | 0.9875 |
| **v2** asc+desc, random_clouds | Occluded | 916.5 | 1128.8 | 22.04 | 0.4229 | 0.0965 | 0.6420 |
| **v2** asc+desc, random_clouds | Observed | 66.2 | 96.1 | 40.52 | 0.8095 | 0.0282 | 0.9894 |
| **v3** mix_closest, random_clouds | Occluded | 406.1 | 585.0 | 25.54 | 0.3765 | 0.1065 | 0.7422 |
| **v3** mix_closest, random_clouds | Observed | 68.9 | 100.0 | 40.19 | 0.8142 | 0.0298 | 0.9883 |

### Masquage : Consecutive Fully Masked

| Modèle | Type | MAE | RMSE | PSNR | SSIM | SAM | R2 |
|:---|:---|:---:|:---:|:---:|:---:|:---:|:---:|
| **ALL_SAR_120_epochs** (mix_closest) | Occluded | 1081.9 | 1366.0 | 19.35 | 0.2190 | 0.1284 | 0.4716 |
| **ALL_SAR_120_epochs** (mix_closest) | Observed | 69.4 | 99.4 | 40.16 | 0.8098 | 0.0289 | 0.9910 |
| asc+desc, random_clouds, DA | Occluded | 2296.9 | 2595.1 | 16.45 | 0.1560 | 0.6064 | 0.0721 |
| asc+desc, random_clouds, DA | Observed | 64.5 | 94.2 | 40.67 | 0.8180 | 0.0271 | 0.9905 |
| asc+desc, random_clouds | Occluded | 2266.2 | 2570.1 | 17.09 | 0.1581 | 0.8594 | 0.0573 |
| asc+desc, random_clouds | Observed | 56.3 | 82.3 | 41.81 | 0.8445 | 0.0234 | 0.9932 |
| asc+desc, random_fully_masked, DA | Occluded | 1157.0 | 1512.9 | 19.39 | 0.2904 | 0.1122 | 0.5013 |
| asc+desc, random_fully_masked, DA | Observed | 67.9 | 98.5 | 40.31 | 0.8272 | 0.0290 | 0.9890 |
| mix_closest, random_fully_masked | Occluded | 1305.4 | 1650.9 | 18.62 | 0.2937 | 0.1147 | 0.4997 |
| mix_closest, random_fully_masked | Observed | 71.3 | 103.1 | 39.92 | 0.8185 | 0.0300 | 0.9877 |
| mix_closest, random_fully_masked, DA | Occluded | 1437.6 | 1822.1 | 17.70 | 0.2879 | 0.1353 | 0.4253 |
| mix_closest, random_fully_masked, DA | Observed | 65.1 | 94.9 | 40.60 | 0.8332 | 0.0264 | 0.9912 |
| coherence_only | Occluded | 2309.3 | 2608.6 | 15.12 | 0.1041 | 0.5966 | 0.1403 |
| coherence_only | Observed | 71.6 | 104.2 | 39.79 | 0.8038 | 0.0300 | 0.9883 |
| **v2** mix_closest, random_fully_masked | Occluded | 1195.6 | 1559.4 | 19.31 | 0.3089 | 0.1106 | 0.5010 |
| **v2** mix_closest, random_fully_masked | Observed | 73.7 | 106.7 | 39.60 | 0.8166 | 0.0311 | 0.9867 |
| **v2** mix_closest, random_clouds | Occluded | 542.1 | 847.0 | 22.71 | 0.2766 | 0.1151 | 0.5809 |
| **v2** mix_closest, random_clouds | Observed | 59.9 | 85.8 | 41.47 | 0.8300 | 0.0249 | 0.9922 |
| **v2** asc+desc, random_fully_masked | Occluded | 1192.1 | 1559.4 | 19.13 | 0.3149 | 0.1111 | 0.4921 |
| **v2** asc+desc, random_fully_masked | Observed | 71.9 | 104.1 | 39.82 | 0.8117 | 0.0302 | 0.9875 |
| **v2** asc+desc, random_clouds | Occluded | 1055.4 | 1341.4 | 19.59 | 0.2374 | 0.1191 | 0.4830 |
| **v2** asc+desc, random_clouds | Observed | 66.4 | 96.4 | 40.48 | 0.8088 | 0.0283 | 0.9893 |
| **v3** mix_closest, random_clouds | Occluded | 626.4 | 907.3 | 21.91 | 0.2504 | 0.1291 | 0.5601 |
| **v3** mix_closest, random_clouds | Observed | 69.2 | 100.4 | 40.15 | 0.8135 | 0.0298 | 0.9883 |

## 4. Meilleurs modèles — Métriques Occluded

### Random Fully Masked

#### Classement (pixels occluded, score = rang moyen pondéré)

| # | Modèle | Score | MAE | RMSE | PSNR | SSIM | SAM | R2 |
|:---:|:---|:---:|:---:|:---:|:---:|:---:|:---:|:---:|
| 1 | **v2** mix_closest, random_clouds | 1.44 | 360.5 | 538.3 | 26.36 | 0.4566 | 0.0903 | 0.7770 |
| 2 | **ALL_SAR_120_epochs** (mix_closest) | 3.22 | 899.0 | 1108.3 | 22.54 | 0.4409 | 0.0886 | 0.6645 |
| 3 | **v3** mix_closest, random_clouds | 3.78 | 406.1 | 585.0 | 25.54 | 0.3765 | 0.1065 | 0.7422 |
| 4 | **v2** asc+desc, random_clouds | 4.67 | 916.5 | 1128.8 | 22.04 | 0.4229 | 0.0965 | 0.6420 |
| 5 | **v2** asc+desc, random_fully_masked | 7.11 | 1070.0 | 1343.3 | 21.33 | 0.4591 | 0.0980 | 0.6270 |

#### Détail par bande du meilleur modèle : **v2** mix_closest, random_clouds

| Métrique | B2 | B3 | B4 | B5 | B6 | B7 | B8 | B8A | B11 | B12 |
|:---|:---:|:---:|:---:|:---:|:---:|:---:|:---:|:---:|:---:|:---:|
| MAE | 209.0 | 221.9 | 292.8 | 284.3 | 400.8 | 478.0 | 509.1 | 480.7 | 375.2 | 353.0 |
| RMSE | 335.6 | 349.1 | 447.9 | 424.4 | 550.2 | 644.7 | 679.5 | 645.2 | 512.2 | 485.9 |
| PSNR | 32.55 | 31.66 | 28.77 | 29.24 | 26.16 | 24.65 | 24.11 | 24.60 | 26.52 | 27.09 |
| SSIM | 0.3661 | 0.4102 | 0.4067 | 0.4740 | 0.4865 | 0.4885 | 0.4686 | 0.4997 | 0.4937 | 0.4721 |
| R2 | 0.4490 | 0.5062 | 0.4971 | 0.5012 | 0.5062 | 0.5218 | 0.5210 | 0.5419 | 0.5152 | 0.5252 |

### Consecutive Fully Masked

#### Classement (pixels occluded, score = rang moyen pondéré)

| # | Modèle | Score | MAE | RMSE | PSNR | SSIM | SAM | R2 |
|:---:|:---|:---:|:---:|:---:|:---:|:---:|:---:|:---:|
| 1 | **v2** mix_closest, random_clouds | 2.00 | 542.1 | 847.0 | 22.71 | 0.2766 | 0.1151 | 0.5809 |
| 2 | **v3** mix_closest, random_clouds | 3.22 | 626.4 | 907.3 | 21.91 | 0.2504 | 0.1291 | 0.5601 |
| 3 | asc+desc, random_fully_masked, DA | 4.11 | 1157.0 | 1512.9 | 19.39 | 0.2904 | 0.1122 | 0.5013 |
| 4 | **v2** mix_closest, random_fully_masked | 4.78 | 1195.6 | 1559.4 | 19.31 | 0.3089 | 0.1106 | 0.5010 |
| 5 | **v2** asc+desc, random_clouds | 4.78 | 1055.4 | 1341.4 | 19.59 | 0.2374 | 0.1191 | 0.4830 |

#### Détail par bande du meilleur modèle : **v2** mix_closest, random_clouds

| Métrique | B2 | B3 | B4 | B5 | B6 | B7 | B8 | B8A | B11 | B12 |
|:---|:---:|:---:|:---:|:---:|:---:|:---:|:---:|:---:|:---:|:---:|
| MAE | 387.5 | 395.8 | 456.7 | 463.1 | 614.7 | 695.0 | 727.2 | 709.5 | 516.4 | 454.6 |
| RMSE | 741.8 | 717.1 | 771.7 | 762.6 | 874.1 | 953.4 | 987.4 | 965.6 | 691.7 | 614.9 |
| PSNR | 26.04 | 25.89 | 24.64 | 24.59 | 22.42 | 21.42 | 21.04 | 21.23 | 23.70 | 24.76 |
| SSIM | 0.2018 | 0.2304 | 0.2403 | 0.2770 | 0.2991 | 0.3060 | 0.2901 | 0.3136 | 0.3112 | 0.2961 |
| R2 | 0.2062 | 0.2555 | 0.3012 | 0.2817 | 0.2395 | 0.2634 | 0.2745 | 0.2748 | 0.3254 | 0.3512 |

## 5. Disponibilité des résultats

| Modèle | Random Fully Masked | Consecutive Fully Masked |
|:---|:---:|:---:|
| **ALL_SAR_120_epochs** (mix_closest) | ✅ | ✅ |
| asc+desc, random_clouds, DA | ✅ | ✅ |
| asc+desc, random_clouds | ✅ | ✅ |
| asc+desc, random_fully_masked, DA | ✅ | ✅ |
| mix_closest, random_fully_masked | ✅ | ✅ |
| mix_closest, random_fully_masked, DA | ✅ | ✅ |
| coherence_only | ✅ | ✅ |
| **v2** mix_closest, random_fully_masked | ✅ | ✅ |
| **v2** mix_closest, random_clouds | ✅ | ✅ |
| **v2** asc+desc, random_fully_masked | ✅ | ✅ |
| **v2** asc+desc, random_clouds | ✅ | ✅ |
| **v3** mix_closest, random_clouds | ✅ | ✅ |

## 6. Métriques à la Parcelle (RPG)

### Masquage : Random Fully Masked

| Modèle | Type | MAE | RMSE | PSNR | SSIM | SAM | R2 |
|:---|:---|:---:|:---:|:---:|:---:|:---:|:---:|
| **ALL_SAR_120_epochs** (mix_closest) | Global | 1116.8 | 2113.2 | 21.58 | 0.6394 | 0.0739 | 0.4569 |
| **ALL_SAR_120_epochs** (mix_closest) | Occluded | 6419.1 | 6743.0 | 3.59 | 0.0806 | 0.3005 | 0.0218 |
| **ALL_SAR_120_epochs** (mix_closest) | Observed | 70.2 | 100.3 | 40.08 | 0.8085 | 0.0287 | 0.9912 |
| **v2** mix_closest, random_clouds | Global | — | — | — | — | — | — |
| **v2** mix_closest, random_clouds | Occluded | — | — | — | — | — | — |
| **v2** mix_closest, random_clouds | Observed | — | — | — | — | — | — |

### Masquage : Consecutive Fully Masked

| Modèle | Type | MAE | RMSE | PSNR | SSIM | SAM | R2 |
|:---|:---|:---:|:---:|:---:|:---:|:---:|:---:|
| **ALL_SAR_120_epochs** (mix_closest) | Global | — | — | — | — | — | — |
| **ALL_SAR_120_epochs** (mix_closest) | Occluded | — | — | — | — | — | — |
| **ALL_SAR_120_epochs** (mix_closest) | Observed | — | — | — | — | — | — |
| **v2** mix_closest, random_clouds | Global | — | — | — | — | — | — |
| **v2** mix_closest, random_clouds | Occluded | — | — | — | — | — | — |
| **v2** mix_closest, random_clouds | Observed | — | — | — | — | — | — |

_Aucun résultat parcelle disponible._


## 7. Impact du Filtrage des Pixels Noirs (nodata)

Comparaison des métriques **avant** et **après** exclusion des pixels où toutes les bandes = 0 dans la cible.

| Modèle | Masquage | Version | MAE | RMSE | PSNR | SSIM | SAM | R2 |
|:---|:---|:---|:---:|:---:|:---:|:---:|:---:|:---:|
| **ALL_SAR_120_epochs** (mix_closest) | Random Fully Masked | Sans filtrage | 6938.1 | 7236.0 | 3.13 | 0.0806 | 0.2969 | 0.0242 |
| **ALL_SAR_120_epochs** (mix_closest) | Random Fully Masked | Avec filtrage | 899.0 | 1108.3 | 22.54 | 0.4409 | 0.0886 | 0.6645 |
| **ALL_SAR_120_epochs** (mix_closest) | Consecutive Fully Masked | Sans filtrage | 7029.8 | 7273.2 | 3.05 | 0.0820 | 0.2818 | 0.0303 |
| **ALL_SAR_120_epochs** (mix_closest) | Consecutive Fully Masked | Avec filtrage | 1081.9 | 1366.0 | 19.35 | 0.2190 | 0.1284 | 0.4716 |
